# **Business Case:**
#### A rapidly growing real estate company is facing challenges in monitoring sales performance, marketing effectiveness, employee productivity, and operational expenses. Decision-makers lack a centralized system to track KPIs and evaluate business performance.
#### The objective of this project is to build an end-to-end Data Analytics solution using Python, SQL Server, and Power BI to support data-driven decisions.

# **📑Project Notebook: Synthetic Data Generation Engine**

# Project Folder:
### New_Cairo_RealEstate_DW
# Pipeline:
###Python Generation $\rightarrow$ Validation $\rightarrow$ Live Google Sheets Export $\rightarrow$ Export CSVs Files

#### **0a. Google Colab Environment & Google Drive Authentication**
###### Mounts Google Drive and authenticates gspread to allow direct read/write access to Google Sheets within the project folder.

In [25]:
from google.colab import drive, auth
import gspread
from google.auth import default
from googleapiclient.discovery import build

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Authenticate
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

print("✅ Authentication successful! Connected to Google Drive & Sheets API.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Authentication successful! Connected to Google Drive & Sheets API.


#### **0b. Core Libraries Setup**
###### Imports necessary modules for synthetic data creation and data manipulation (pandas, random).

In [26]:
import random
import pandas as pd
from datetime import datetime, timedelta

# Set global seed for reproducible synthetic generation
random.seed(42)
print("✅ Core Python libraries loaded.")

✅ Core Python libraries loaded.


#### **0c. Work Schedule Calendar and Data Department Staff Map:**
###### Weekly days off in Egypt, Official Holidays (2023–2026), Fixed holidays, Approximate movable holidays, Official working hours

In [27]:
from datetime import datetime, timedelta, date
import random

# ============================================================
# Joint Action Calendar:(Properties, Sellers, Marketing Leads, Operations)
# ============================================================

# 1. Weekly days off in Egypt: Friday and Saturday.
#    (Python: Monday=0 ... Friday=4, Saturday=5, Sunday=6)
WEEKEND_DAYS = {4, 5}

# 2. Official Holidays (2023–2026) — Fixed dates + approximate dates for Hijri holidays
def _expand_range(start_str, end_str):
    start = datetime.strptime(start_str, '%Y-%m-%d').date()
    end = datetime.strptime(end_str, '%Y-%m-%d').date()
    return [start + timedelta(days=i) for i in range((end - start).days + 1)]

HOLIDAYS = set()
holiday_ranges = [
    # --- Fixed holidays (recurring on the same date every year) ---
    ('2023-01-01', '2023-01-01'), ('2023-01-07', '2023-01-07'), ('2023-01-25', '2023-01-25'),
    ('2023-04-25', '2023-04-25'), ('2023-05-01', '2023-05-01'), ('2023-06-30', '2023-06-30'),
    ('2023-07-23', '2023-07-23'), ('2023-10-06', '2023-10-06'),
    ('2024-01-01', '2024-01-01'), ('2024-01-07', '2024-01-07'), ('2024-01-25', '2024-01-25'),
    ('2024-04-25', '2024-04-25'), ('2024-05-01', '2024-05-01'), ('2024-06-30', '2024-06-30'),
    ('2024-07-23', '2024-07-23'), ('2024-10-06', '2024-10-06'),
    ('2025-01-01', '2025-01-01'), ('2025-01-07', '2025-01-07'), ('2025-01-25', '2025-01-25'),
    ('2025-04-25', '2025-04-25'), ('2025-05-01', '2025-05-01'), ('2025-06-30', '2025-06-30'),
    ('2025-07-23', '2025-07-23'), ('2025-10-06', '2025-10-06'),
    ('2026-01-01', '2026-01-01'), ('2026-01-07', '2026-01-07'), ('2026-01-25', '2026-01-25'),
    ('2026-04-25', '2026-04-25'), ('2026-05-01', '2026-05-01'),

    # --- Approximate movable holidays (Islamic holidays + Sham Ennessim)---
    ('2023-04-17', '2023-04-17'),                    # شم النسيم
    ('2023-04-21', '2023-04-23'),                    # عيد الفطر
    ('2023-06-28', '2023-07-01'),                    # عيد الأضحى
    ('2023-07-19', '2023-07-19'),                    # رأس السنة الهجرية
    ('2023-09-27', '2023-09-27'),                    # المولد النبوي
    ('2024-05-06', '2024-05-06'),                    # شم النسيم
    ('2024-04-10', '2024-04-12'),                    # عيد الفطر
    ('2024-06-16', '2024-06-19'),                    # عيد الأضحى
    ('2024-07-07', '2024-07-07'),                    # رأس السنة الهجرية
    ('2024-09-15', '2024-09-15'),                    # المولد النبوي
    ('2025-04-21', '2025-04-21'),                    # شم النسيم
    ('2025-03-30', '2025-04-01'),                    # عيد الفطر
    ('2025-06-06', '2025-06-09'),                    # عيد الأضحى
    ('2025-06-26', '2025-06-26'),                    # رأس السنة الهجرية
    ('2025-09-04', '2025-09-04'),                    # المولد النبوي
    ('2026-04-13', '2026-04-13'),                    # شم النسيم
    ('2026-03-20', '2026-03-22'),                    # عيد الفطر (تقريبي)
]
for start_str, end_str in holiday_ranges:
    HOLIDAYS.update(_expand_range(start_str, end_str))

# 3. Official working hours: From 9:00 AM to 5:00 PM.
WORK_START_HOUR = 9
WORK_END_HOUR = 17

def is_working_day(check_date):
    """Returns True if the day is an actual workday (neither a weekend nor a public holiday)"""
    if check_date.weekday() in WEEKEND_DAYS:
        return False
    if check_date in HOLIDAYS:
        return False
    return True

# 4. Map of Data Management staff who actually enter unit and vendor data
#    (Actual work periods for each employee based on Dim_Employees)
DATA_ENTRY_STAFF_WINDOWS = {
    'EMP_003': (datetime(2023, 1, 15), None),                      # Nouran Ali - Sales Admin Senior، In Active
    'EMP_004': (datetime(2023, 2, 1), None),                       # Kareem Fahmy - Moderator، In Active
    'EMP_020': (datetime(2025, 2, 1), None),                       # Mariam Hossam - Sales Admin Junior، In Active
    'EMP_021': (datetime(2025, 4, 1), datetime(2025, 10, 31)),     # Ramy Wagdy - Data Collector، Terminated
}

def pick_available_employee(windows, check_datetime):
    """It returns a random employee from among those who were actually working at that specific date and timث"""
    available = [
        eid for eid, (start, end) in windows.items()
        if start <= check_datetime and (end is None or check_datetime <= end)
    ]
    if not available:
        available = [eid for eid, (start, end) in windows.items() if end is None]
    return random.choice(available)

def generate_work_datetime(year, month=None):
    """
    It generates a truly random date and time falling on an actual workday (not a weekend or holiday) within official working hours, and returns the employee who entered the data at that time
    """
    for _ in range(200): # Enough attempts to find a proper workday
        m = month if month else random.randint(1, 12)
        try:
            random_day = date(year, m, random.randint(1, 28))
        except ValueError:
            continue

        if is_working_day(random_day):
            hour = random.randint(WORK_START_HOUR, WORK_END_HOUR - 1)
            minute = random.randint(0, 59)
            result_datetime = datetime.combine(random_day, datetime.min.time()) + timedelta(hours=hour, minutes=minute)
            entered_by = pick_available_employee(DATA_ENTRY_STAFF_WINDOWS, result_datetime)
            return result_datetime, entered_by

# Backup: If we don't find a day's work after all attempts (very rare)
    fallback = datetime(year, 6, 15, 11, 0)
    return fallback, pick_available_employee(DATA_ENTRY_STAFF_WINDOWS, fallback)

print(f"✅ تقويم العمل جاهز: {len(HOLIDAYS)} Registered official holiday, working hours {WORK_START_HOUR}:00 - {WORK_END_HOUR}:00.")

✅ تقويم العمل جاهز: 62 Registered official holiday, working hours 9:00 - 17:00.


## **• Target Table:** Dim_Properties •



#### **1. Configures the geographic breakdown for New Cairo (NC) across years (2023–2026**)
###### Distinctly separates Square, District, and Zone for accurate relational querying in SQL Server and Power BI.

In [28]:
REGION_CODE = 'NC'  # Scalable prefix for New Cairo

GEOGRAPHIC_MAP = {
    2023: {
        'Square': '1st Settlement',
        'Category': 'Open Area',
        'CategoryCode': '2',
        'Data': [
            {'District': 'Banafseg', 'Zone': '1'},
            {'District': 'Banafseg', 'Zone': '2'},
            {'District': 'Banafseg', 'Zone': '3'},
            {'District': 'Banafseg', 'Zone': '4'},
            {'District': 'Banafseg', 'Zone': '5'},
            {'District': 'Banafseg', 'Zone': '6'},
            {'District': 'Banafseg', 'Zone': '7'},
            {'District': 'Banafseg', 'Zone': '8'},
            {'District': 'Banafseg', 'Zone': '9'},
            {'District': 'Banafseg', 'Zone': '10'},
            {'District': 'Banafseg', 'Zone': '11'},
            {'District': 'Banafseg', 'Zone': '12'},
            {'District': 'Banafseg', 'Zone': 'Omarat'},
            {'District': 'Al Banafseg', 'Zone': 'X-Mall'}
        ]
    },
    2024: {
        'Square': '1st Settlement',
        'Category': 'Compound',
        'CategoryCode': '1',
        'Data': [
            {'District': 'Creek Town', 'Zone': 'IL Cazar'},
            {'District': 'Mirage Residence', 'Zone': 'Main'},
            {'District': 'Nakheel', 'Zone': 'Resort'}
        ]
    },
    2025: {
        'Square': '5th Settlement',
        'Category': 'Open Area',
        'CategoryCode': '2',
        'Data': [
            {'District': 'Hay', 'Zone': '1'},
            {'District': 'Hay', 'Zone': '2'},
            {'District': 'Hay', 'Zone': '3'},
            {'District': 'Hay', 'Zone': '4'},
            {'District': 'Hay', 'Zone': '5'},
            {'District': 'Hay 5', 'Zone': 'Buildings'},
            {'District': 'Silver Star Mall', 'Zone': 'H5a'}
        ]
    },
    2026: {
        'Square': '5th Settlement',
        'Category': 'Compound',
        'CategoryCode': '1',
        'Data': [
            {'District': 'Sodic Eastown', 'Zone': 'EDNC'},
            {'District': 'Sodic Villette', 'Zone': 'Main'},
            {'District': 'Sodic Eastown', 'Zone': 'Main'},
            {'District': 'Mivida', 'Zone': 'Main'},
            {'District': 'Mivida', 'Zone': 'Boulevard'},
            {'District': 'Mivida Business Park', 'Zone': 'Mall'}
        ]
    }
}
print("✅ Geographic taxonomy mapping loaded.")

✅ Geographic taxonomy mapping loaded.


#### **2. Property Taxonomy & Encoding Rules**
###### Defines unit usage types (Residential, Administrative, Commercial, Medical) and structural coding scheme for PropertyID: {Region}_{CategoryCode}{UsageCategoryLetter}{PropertyTypeLetter}{Sequence} (e.g., NC_1aA0000001).

In [29]:
PROPERTY_TYPES = [
    # Residential (a)
    {'UsageCode': 'a', 'TypeCode': 'A', 'Name': 'Apartment', 'Usage': 'Residential'},
    {'UsageCode': 'a', 'TypeCode': 'B', 'Name': 'Studio', 'Usage': 'Residential'},
    {'UsageCode': 'a', 'TypeCode': 'C', 'Name': 'Duplex', 'Usage': 'Residential'},
    {'UsageCode': 'a', 'TypeCode': 'D', 'Name': 'Townhouse', 'Usage': 'Residential'},

    # Administrative (b)
    {'UsageCode': 'b', 'TypeCode': 'A', 'Name': 'Office', 'Usage': 'Administrative'},

    # Commercial (c)
    {'UsageCode': 'c', 'TypeCode': 'A', 'Name': 'Store/Market', 'Usage': 'Commercial'},
    {'UsageCode': 'c', 'TypeCode': 'B', 'Name': 'Storage', 'Usage': 'Commercial'},

    # Medical (d)
    {'UsageCode': 'd', 'TypeCode': 'A', 'Name': 'Clinic', 'Usage': 'Medical'},
    {'UsageCode': 'd', 'TypeCode': 'B', 'Name': 'Pharmacy', 'Usage': 'Medical'}
]

SELLER_TYPES = ['OW', 'BR', 'HS', 'DV']

# بناء خريطة الاستخدام تلقائيًا: usage_code -> (اسم الاستخدام, {type_code: اسم النوع})
USAGE_MAP = {}
for pt in PROPERTY_TYPES:
    code = pt['UsageCode']
    if code not in USAGE_MAP:
        USAGE_MAP[code] = (pt['Usage'], {})
    USAGE_MAP[code][1][pt['TypeCode']] = pt['Name']

print("✅ Property taxonomy and seller codes initialized.")

✅ Property taxonomy and seller codes initialized.


#### **3. Domain Logic Engine (Finishing, Furnishing & Commercial Options)**
###### Enforces domain rules to ensure consistency:Fully Finished $\rightarrow$ Allowed Furnished/Semi/Unfurnished + Rent/Sale Cash/Installments.Semi Finished / Core & Shell $\rightarrow$ Must be Unfurnished and strictly Sale Cash/Installment (No Rentals).

In [30]:
def apply_property_domain_rules(finishing, usage):
    """
    Enforces business constraints between finishing status, furnishing, and deal availability.
    """
    if finishing in ['Core & Shell', 'Semi Finished']:
        furnishing = 'Unfurnished'
        allowed_deals = ['Sale Cash', 'Sale Installment']
    else:  # Fully Finished
        furnishing = random.choice(['Unfurnished', 'Semi Furnished', 'Fully Furnished'])
        allowed_deals = ['Sale Cash', 'Sale Installment', 'Rent']

    deal_type = random.choice(allowed_deals)
    return furnishing, deal_type

#### **4. Synthetic Generation Engine**
###### Runs the iteration loop to construct Dim_Properties dataframe enforcing floor constraints, BUA calculations, room setups, downpayments, and remaining balances.

In [31]:
def generate_properties_dataset(num_records=500):
    records = []

    FLOORS = ['Ground', '1', '2', '3', '4', '5', '6', '7', '8', 'Roof']
    BUA_OPTIONS = [65, 80, 100, 120, 150, 180, 220, 280, 350, 450, 500]
    AMENITIES_POOL = ['Elevator', 'Parking', 'Garden', 'Roof', 'Pool']
    FINISHING_TYPES = ['Core & Shell', 'Semi Finished', 'Fully Finished']
    PROJECT_END_DATE = datetime(2026, 5, 1)

    for i in range(1, num_records + 1):
        # 1. اختيار السنة، وبالتبعية Square / District / Zone / PropertyCategory
        year = random.choice(list(GEOGRAPHIC_MAP.keys()))
        geo_info = GEOGRAPHIC_MAP[year]
        square = geo_info['Square']
        property_category = geo_info['Category']
        category_code = geo_info['CategoryCode']
        location = random.choice(geo_info['Data'])
        district = location['District']
        zone = location['Zone']

        # 2. اختيار الاستخدام ونوع الوحدة
        usage_code = random.choice(list(USAGE_MAP.keys()))
        usage_name, type_map = USAGE_MAP[usage_code]
        prop_type_code = random.choice(list(type_map.keys()))
        prop_type_name = type_map[prop_type_code]

        # 3. تركيب PropertyID
        property_id = f"{REGION_CODE}_{category_code}{usage_code}{prop_type_code}{i:07d}"

        # 4. اختيار البائع
        seller_type_code = random.choice(SELLER_TYPES)
        seller_num = random.randint(1, 200)
        seller_id = f"{seller_type_code}{seller_num:07d}"

        # 5. قواعد التشطيب/الفرش/طريقة البيع حسب نوع البائع
        finishing = random.choice(FINISHING_TYPES)
        if seller_type_code == 'DV':
            furnishing = 'Unfurnished'
            deal_type = random.choice(['Sale Cash', 'Sale Installment'])
        elif seller_type_code == 'HS':
            finishing = 'Fully Finished'
            furnishing = random.choice(['Semi Furnished', 'Fully Furnished'])
            deal_type = 'Rent'
        else:
            furnishing, deal_type = apply_property_domain_rules(finishing, usage_name)

        # 6. الطابق والمساحة
        floor = random.choice(FLOORS)
        bua_sqm = random.choice(BUA_OPTIONS)

        # 7. الملحقات
        num_amenities = random.randint(0, 3)
        amenities = ', '.join(random.sample(AMENITIES_POOL, num_amenities)) if num_amenities > 0 else None

        # 8. الغرف والحمامات
        if usage_code == 'a':
            master_room = random.randint(0, 1)
            main_room = random.randint(1, 4)
            maids_room = random.randint(0, 1)
            master_bathroom = random.randint(0, 1)
            main_bathroom = random.randint(1, 3)
        else:
            master_room = 0
            main_room = 1
            maids_room = 0
            master_bathroom = 0
            main_bathroom = 1
        guest_toilet = random.randint(0, 1)

        # 9. الحسابات المالية حسب نوع الصفقة
        if deal_type == 'Rent':
            rate_per_sqm = random.randint(200, 500)
            amount_downpayment = round(bua_sqm * rate_per_sqm, -3)
            remaining_amount = 0
            unit_status = random.choice(['Rented', 'Available', 'Hold Temporarily', 'Unreachable'])
        else:
            rate_per_sqm = random.randint(30000, 70000)
            total_price = round(bua_sqm * rate_per_sqm, -4)
            if deal_type == 'Sale Installment':
                amount_downpayment = round(total_price * random.choice([0.10, 0.15, 0.20]), -4)
                remaining_amount = total_price - amount_downpayment
            else:
                amount_downpayment = total_price
                remaining_amount = 0
            unit_status = random.choice(['Sold', 'Available', 'Hold Temporarily', 'Unreachable'])

        # 10. 🆕 تاريخ ووقت الإدخال الفعلي + الموظف إدخال البيانات
        created_datetime, entered_by = generate_work_datetime(year)

        # 11. 🆕 تاريخ التأجير (للوحدات المؤجرة فعليًا)
        rented_datetime = None
        if deal_type == 'Rent' and unit_status == 'Rented':
            days_to_rent = random.randint(3, 60)
            candidate_rent_date = created_datetime + timedelta(days=days_to_rent)
            rented_datetime = min(candidate_rent_date, PROJECT_END_DATE)

        records.append({
            'PropertyID': property_id,
            'SellerID': seller_id,
            'Square': square,
            'District': district,
            'Zone': zone,
            'YearAdded': year,
            'PropertyCategory': property_category,
            'UsageType': usage_name,
            'PropertyType': prop_type_name,
            'Floor': floor,
            'BUA_SQM': bua_sqm,
            'Amenities': amenities,
            'MasterRoom': master_room,
            'MainRoom': main_room,
            'MaidsRoom': maids_room,
            'MasterBathroom': master_bathroom,
            'MainBathroom': main_bathroom,
            'GuestToilet': guest_toilet,
            'FinishingLevel': finishing,
            'FurnishingStatus': furnishing,
            'SellingOption': deal_type,
            'UnitStatus': unit_status,
            'Amount_Downpayment': amount_downpayment,
            'RemainingAmount': remaining_amount,
            'CreatedDateTime': created_datetime.strftime('%Y-%m-%d %H:%M:%S'),
            'EnteredByEmployeeID': entered_by,
            'RentedDateTime': rented_datetime.strftime('%Y-%m-%d %H:%M:%S') if rented_datetime else ''
        })

    return pd.DataFrame(records)

# تشغيل التوليد
df_dim_properties = generate_properties_dataset(num_records=500)
print(f"🎉 Generated {len(df_dim_properties)} records for Dim_Properties (مع تواريخ إدخال دقيقة بالدقيقة)!")

🎉 Generated 500 records for Dim_Properties (مع تواريخ إدخال دقيقة بالدقيقة)!


#### **5. Integrity check:** Entered By EmployeeID

In [32]:
print("=== توزيع التواريخ حسب السنة ===")
df_dim_properties['CreatedDateTime'] = pd.to_datetime(df_dim_properties['CreatedDateTime'])
print(df_dim_properties['CreatedDateTime'].dt.year.value_counts().sort_index())

rented_count = df_dim_properties[df_dim_properties['RentedDateTime'] != '']
print(f"\n🔍 عدد الوحدات التي لها RentedDateTime = {len(rented_count)} (المفروض يساوي عدد الوحدات UnitStatus='Rented')")

=== توزيع التواريخ حسب السنة ===
CreatedDateTime
2023    115
2024    132
2025    119
2026    134
Name: count, dtype: int64

🔍 عدد الوحدات التي لها RentedDateTime = 37 (المفروض يساوي عدد الوحدات UnitStatus='Rented')


#### **6. Data Validation & Sanity Inspection**
###### Inspects data samples and validates zero business rule violations before cloud sync.

In [33]:
# معاينة عينة من البيانات
print("=== Dim_Properties Sample Preview ===")
display(df_dim_properties[['PropertyID', 'SellerID', 'Square', 'District', 'PropertyType',
                            'SellingOption', 'BUA_SQM', 'Amount_Downpayment',
                            'RemainingAmount', 'UnitStatus']].head(10))

# فحص 1: لا توجد قيم سالبة في المتبقي من السعر
negative_remaining = df_dim_properties[df_dim_properties['RemainingAmount'] < 0]

# فحص 2: قاعدة (تشطيب غير مكتمل + إيجار) غير ممكنة الحدوث
unfinished_rentals = df_dim_properties[
    (df_dim_properties['FinishingLevel'].isin(['Core & Shell', 'Semi Finished'])) &
    (df_dim_properties['SellingOption'] == 'Rent')
]

print("\n--- Sanity Inspection Audit ---")
print(f"✅ قيم RemainingAmount سالبة: {len(negative_remaining)}")
print(f"✅ خرق لقاعدة (تشطيب غير مكتمل + إيجار): {len(unfinished_rentals)}")

=== Dim_Properties Sample Preview ===


,PropertyID,SellerID,Square,District,PropertyType,SellingOption,BUA_SQM,Amount_Downpayment,RemainingAmount,UnitStatus
0,NC_2cA0000001,BR0000036,1st Settlement,Banafseg,Store/Market,Rent,80,26000.0,0.0,Rented
1,NC_2aB0000002,DV0000088,5th Settlement,Silver Star Mall,Studio,Sale Cash,180,5910000.0,0.0,Unreachable
2,NC_1aA0000003,BR0000198,1st Settlement,Nakheel,Apartment,Sale Cash,80,2770000.0,0.0,Available
3,NC_1bA0000004,OW0000059,5th Settlement,Sodic Eastown,Office,Sale Installment,150,2020000.0,8060000.0,Hold Temporarily
4,NC_1cA0000005,BR0000191,5th Settlement,Sodic Villette,Store/Market,Sale Installment,220,790000.0,7120000.0,Sold
5,NC_1dB0000006,OW0000175,5th Settlement,Mivida,Pharmacy,Rent,150,42000.0,0.0,Unreachable
6,NC_2bA0000007,HS0000196,5th Settlement,Silver Star Mall,Office,Rent,180,58000.0,0.0,Rented
7,NC_1dA0000008,HS0000136,1st Settlement,Creek Town,Clinic,Rent,350,137000.0,0.0,Unreachable
8,NC_2bA0000009,OW0000019,1st Settlement,Banafseg,Office,Sale Cash,80,2770000.0,0.0,Available
9,NC_1dB0000010,BR0000025,5th Settlement,Sodic Villette,Pharmacy,Sale Installment,220,1120000.0,6350000.0,Hold Temporarily



--- Sanity Inspection Audit ---
✅ قيم RemainingAmount سالبة: 0
✅ خرق لقاعدة (تشطيب غير مكتمل + إيجار): 0


#### **7. Google Sheets Sync Pipeline**
###### Finds the target Drive folder New_Cairo_RealEstate_DW, creates or connects to NC_RealEstate_DW_Data Google Sheet, and exports Dim_Properties into a clean dedicated tab.

In [34]:
FOLDER_NAME = "New_Cairo_RealEstate_DW"
SPREADSHEET_NAME = "NC_RealEstate_DW_Data"
TAB_NAME = "Dim_Properties"

drive_service = build('drive', 'v3', credentials=creds)

# 1. Locate Target Folder ID in Drive
results = drive_service.files().list(
    q=f"name = '{FOLDER_NAME}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
    fields="files(id, name)"
).execute()
folders = results.get('files', [])

if not folders:
    print(f"⚠️ Folder '{FOLDER_NAME}' not found. Please create it in your Google Drive root.")
else:
    folder_id = folders[0]['id']

    # 2. Open or Create Google Sheet
    try:
        sh = gc.open(SPREADSHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(SPREADSHEET_NAME)
        drive_service.files().update(
            fileId=sh.id,
            addParents=folder_id,
            removeParents='root',
            fields='id, parents'
        ).execute()

    # 3. Create or Select Tab
    try:
        worksheet = sh.worksheet(TAB_NAME)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=TAB_NAME, rows="1000", cols="30")

    # 4. Overwrite Data
    worksheet.clear()
    worksheet.update([df_dim_properties.columns.values.tolist()] + df_dim_properties.astype(str).values.tolist())

    print(f"🚀 Success! [{TAB_NAME}] exported successfully to [{SPREADSHEET_NAME}] inside '{FOLDER_NAME}' folder.")

🚀 Success! [Dim_Properties] exported successfully to [NC_RealEstate_DW_Data] inside 'New_Cairo_RealEstate_DW' folder.


## **• Target Table:** Dim_Clients •

#### **1. Client Demographics & Taxonomy Setup**
#### Defines master taxonomy pools for clients in New Cairo market:
###### • Client Type: Buyer, Seller, Tenant, Landlord.
###### • Lead Source: Social Media (Meta/Instagram), Property Finder, Bayut, Cold Call, Referral, Walk-in, Website.
###### • Preferred Locations: First Settlement, Fifth Settlement, Golden Square, Narges, Banafseg, etc.

In [35]:
CLIENT_TYPES = ['Buyer', 'Seller', 'Tenant', 'Landlord']

LEAD_SOURCES = [
    'Facebook Ads', 'Instagram Ads', 'Property Finder',
    'Bayut/Dubizzle', 'Referral', 'Direct Call / Cold Call',
    'Walk-in Branch', 'Website Form'
]

PREFERRED_LOCATIONS = [
    '1st Settlement - Banafseg', '1st Settlement - Creek Town',
    '5th Settlement - Golden Square', '5th Settlement - Lotus',
    '5th Settlement - Narges', '5th Settlement - Sodic Eastown',
    'Choueifat Area', 'Northern Investors Zone'
]

PREFERRED_USAGE = ['Residential', 'Administrative', 'Commercial', 'Medical']

MALE_NAMES = ['Ahmed', 'Mohamed', 'Mahmoud', 'Omar', 'Khaled', 'Tarek', 'Hassan', 'Youssef', 'Ali', 'Mostafa']
FEMALE_NAMES = ['Nour', 'Mona', 'Sara', 'Dina', 'Aya', 'Reem', 'Habiba', 'Mariam', 'Salma', 'Noha']
LAST_NAMES = ['El-Sayed', 'El-Naggar', 'Abdel-Rahman', 'Ghanem', 'Soliman', 'Amer', 'Fawzy', 'Kassab', 'El-Khatib', 'Shafeek']

print("✅ Client master lookup lists initialized.")

✅ Client master lookup lists initialized.


#### **2. Client Budget Logic**

In [36]:
def generate_client_budget_and_preference(client_type):
    """
    يربط نوع العميل (Buyer/Seller/Tenant/Landlord) بنوع الصفقة ونطاق الميزانية المنطقي.
    Buyer/Seller -> صفقة بيع (Cash أو Installment) وميزانية بالمليون.
    Tenant/Landlord -> صفقة إيجار وميزانية شهرية.
    """
    usage = random.choice(PREFERRED_USAGE)

    if client_type in ['Buyer', 'Seller']:
        deal_type = random.choice(['Sale Cash', 'Sale Installment'])
        if usage == 'Residential':
            budget_min = random.choice([3000000, 5000000, 8000000, 12000000])
            budget_max = budget_min + random.choice([2000000, 5000000, 10000000])
        elif usage in ['Administrative', 'Medical']:
            budget_min = random.choice([4000000, 7000000, 10000000])
            budget_max = budget_min + random.choice([3000000, 6000000])
        else:  # Commercial
            budget_min = random.choice([8000000, 15000000, 25000000])
            budget_max = budget_min + random.choice([5000000, 15000000])
    else:  # Tenant or Landlord
        deal_type = 'Rent'
        if usage == 'Residential':
            budget_min = random.choice([15000, 25000, 40000, 60000])
            budget_max = budget_min + random.choice([10000, 20000, 30000])
        else:  # Commercial / Admin / Medical
            budget_min = random.choice([35000, 60000, 100000, 200000])
            budget_max = budget_min + random.choice([20000, 50000, 100000])

    return usage, deal_type, budget_min, budget_max

print("✅ Client logic engine compiled.")

✅ Client logic engine compiled.


#### **3. Synthetic Generation Engine for Dim_Clients**

In [37]:
random.seed(101)  # Seed خاص ببلوك Dim_Clients (بدل الخلية المحذوفة)

def classify_client(min_budget, usage, deal_type):
    """
    تصنيف العميل لفئة (A/B/C) حسب حجم الميزانية ونوع الاستخدام ونوع الصفقة.
    """
    if deal_type == 'Rent':
        if usage == 'Residential':
            if min_budget >= 50000: return 'Class A'
            elif min_budget >= 25000: return 'Class B'
            else: return 'Class C'
        else:  # Commercial / Admin / Medical
            if min_budget >= 100000: return 'Class A'
            elif min_budget >= 50000: return 'Class B'
            else: return 'Class C'
    else:  # Sale Cash / Sale Installment
        if usage == 'Residential':
            if min_budget >= 8000000: return 'Class A'
            elif min_budget >= 4000000: return 'Class B'
            else: return 'Class C'
        else:  # Commercial / Admin / Medical
            if min_budget >= 15000000: return 'Class A'
            elif min_budget >= 7000000: return 'Class B'
            else: return 'Class C'


def generate_clients_dataset(num_records=500):
    records = []
    start_date = datetime(2023, 1, 1)

    for i in range(1, num_records + 1):
        client_id = f"CL_{i:07d}"

        gender = random.choice(['Male', 'Female'])
        first_name = random.choice(MALE_NAMES) if gender == 'Male' else random.choice(FEMALE_NAMES)
        last_name = random.choice(LAST_NAMES)
        full_name = f"{first_name} {last_name}"

        phone = f"+201{random.choice(['0', '1', '2', '5'])}{random.randint(10000000, 99999999)}"
        email = f"{first_name.lower()}.{last_name.lower()}{random.randint(10, 99)}@gmail.com"

        client_type = random.choice(CLIENT_TYPES)
        lead_source = random.choice(LEAD_SOURCES)
        pref_location = random.choice(PREFERRED_LOCATIONS)

        pref_usage, deal_type, min_budget, max_budget = generate_client_budget_and_preference(client_type)
        client_class = classify_client(min_budget, pref_usage, deal_type)

        random_days = random.randint(0, 1150)
        created_date = start_date + timedelta(days=random_days)
        client_status = random.choice(['Active', 'Converted/Closed', 'Inactive', 'Hot Lead', 'Cold Lead'])

        records.append({
            'ClientID': client_id,
            'ClientName': full_name,
            'Gender': gender,
            'Phone': phone,
            'Email': email,
            'ClientType': client_type,
            'TargetDealType': deal_type,
            'LeadSource': lead_source,
            'PreferredUsage': pref_usage,
            'PreferredLocation': pref_location,
            'MinBudget_EGP': min_budget,
            'MaxBudget_EGP': max_budget,
            'ClientClass': client_class,
            'ClientStatus': client_status,
            'CreatedDate': created_date.strftime('%Y-%m-%d')
        })

    return pd.DataFrame(records)

# تنفيذ التوليد
df_dim_clients = generate_clients_dataset(num_records=500)
print(f"🎉 Generated {len(df_dim_clients)} clean demand-side records for Dim_Clients!")

🎉 Generated 500 clean demand-side records for Dim_Clients!


#### **4. Data Validation & Inspection**

In [38]:
# Preview Sample
print("=== Dim_Clients Sample Preview ===")
display(df_dim_clients[['ClientID', 'ClientName', 'ClientType', 'LeadSource', 'PreferredUsage', 'MinBudget_EGP', 'ClientClass', 'ClientStatus']].head(10))

# Sanity Check Audit
invalid_budgets = df_dim_clients[df_dim_clients['MinBudget_EGP'] > df_dim_clients['MaxBudget_EGP']]
print(f"\n🔍 Audit Result: Total Budget Anomalies = {len(invalid_budgets)}")

=== Dim_Clients Sample Preview ===


,ClientID,ClientName,ClientType,LeadSource,PreferredUsage,MinBudget_EGP,ClientClass,ClientStatus
0,CL_0000001,Ali Amer,Seller,Bayut/Dubizzle,Medical,7000000,Class B,Inactive
1,CL_0000002,Mahmoud El-Naggar,Seller,Website Form,Administrative,7000000,Class B,Hot Lead
2,CL_0000003,Aya Ghanem,Tenant,Referral,Medical,60000,Class B,Hot Lead
3,CL_0000004,Nour Kassab,Tenant,Facebook Ads,Commercial,35000,Class C,Active
4,CL_0000005,Sara Soliman,Tenant,Website Form,Medical,100000,Class A,Active
5,CL_0000006,Mohamed El-Sayed,Landlord,Instagram Ads,Administrative,60000,Class B,Hot Lead
6,CL_0000007,Mohamed Abdel-Rahman,Buyer,Facebook Ads,Medical,4000000,Class C,Active
7,CL_0000008,Sara Soliman,Buyer,Direct Call / Cold Call,Residential,5000000,Class B,Hot Lead
8,CL_0000009,Noha El-Sayed,Landlord,Bayut/Dubizzle,Commercial,35000,Class C,Inactive
9,CL_0000010,Habiba Fawzy,Buyer,Bayut/Dubizzle,Residential,8000000,Class A,Active



🔍 Audit Result: Total Budget Anomalies = 0


#### **5. Google Sheets Export Pipeline**

In [39]:
FOLDER_NAME = "New_Cairo_RealEstate_DW"
SPREADSHEET_NAME = "NC_RealEstate_DW_Data"
TAB_NAME = "Dim_Clients"

drive_service = build('drive', 'v3', credentials=creds)

results = drive_service.files().list(
    q=f"name = '{FOLDER_NAME}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
    fields="files(id, name)"
).execute()
folders = results.get('files', [])

if not folders:
    print(f"⚠️ Folder '{FOLDER_NAME}' not found. Please ensure it exists in Google Drive.")
else:
    folder_id = folders[0]['id']

    try:
        sh = gc.open(SPREADSHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(SPREADSHEET_NAME)
        drive_service.files().update(
            fileId=sh.id,
            addParents=folder_id,
            removeParents='root',
            fields='id, parents'
        ).execute()

    try:
        worksheet = sh.worksheet(TAB_NAME)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=TAB_NAME, rows="1000", cols="30")

    worksheet.clear()
    worksheet.update([df_dim_clients.columns.values.tolist()] + df_dim_clients.astype(str).values.tolist())

    print(f"🚀 Success! Tab [{TAB_NAME}] exported successfully to [{SPREADSHEET_NAME}] inside '{FOLDER_NAME}' folder.")

🚀 Success! Tab [Dim_Clients] exported successfully to [NC_RealEstate_DW_Data] inside 'New_Cairo_RealEstate_DW' folder.


## **• Target Table:** Dim_Sellers •
(Landlords, Developers & Brokers)

#### **1. Seller Taxonomy & Business Rules Setup**
#### Defines seller classifications and enforces business rules for listing options:
###### • Owner (OW) & External Broker (BR): Can list properties for Sale or Rent.
###### • Hospitality (HS): Exclusively offers units for Rent (Serviced/Hotel Apartments).
###### • Developer (DV): Exclusively offers units for Sale (Primary Market / Installments).

In [40]:
SELLER_TYPES = {
    'OW': 'Individual Owner',
    'BR': 'External Co-Broker',
    'HS': 'Hospitality Company',
    'DV': 'Real Estate Developer'
}

DEVELOPER_NAMES = ['Sodic', 'Mivida / Emaar', 'Palm Hills', 'Mountain View', 'ORA Developers', 'Hassan Allam']
HOSPITALITY_NAMES = ['Staybridge Suites', 'Roft Hotel Apartments', 'Chez Residence', 'Cairo Luxury Suites']
BROKER_AGENCIES = ['Coldwell Banker Partner', 'RE/MAX Prime', 'The Address Investments', 'Nawy Partner']

MALE_NAMES = ['Hazem', 'Amr', 'Sherif', 'Wael', 'Nader', 'Hany', 'Ehab', 'Magdy']
FEMALE_NAMES = ['Ghada', 'Rania', 'Riham', 'Engy', 'Nervana', 'Heba', 'May']
LAST_NAMES = ['El-Ghabour', 'El-Kadi', 'Mansour', 'Sawy', 'El-Shazly', 'Farid', 'Bakir']

print("✅ Seller taxonomy and commercial logic loaded.")

✅ Seller taxonomy and commercial logic loaded.


#### **2. Business Logic Engine for Listing Capabilities**
###### Applies constraints mapping seller categories to their valid operational deal types (Sale, Rent, or Both).

In [41]:
def get_seller_business_logic(seller_type_code):
    """
    Enforces side-by-side agent rules, personal individual names for Brokers/Owners,
    and realistic deal-type probability distributions.
    """
    if seller_type_code == 'BR':
        gender = random.choice(['Male', 'Female'])
        first = random.choice(MALE_NAMES) if gender == 'Male' else random.choice(FEMALE_NAMES)
        seller_name = f"{first} {random.choice(LAST_NAMES)} (Co-Broker)"
        allowed_deal_types = random.choices(['Sale', 'Rent', 'Both (Sale & Rent)'], weights=[48, 48, 4])[0]

    elif seller_type_code == 'OW':
        gender = random.choice(['Male', 'Female'])
        first = random.choice(MALE_NAMES) if gender == 'Male' else random.choice(FEMALE_NAMES)
        seller_name = f"{first} {random.choice(LAST_NAMES)}"
        allowed_deal_types = random.choices(['Sale', 'Rent', 'Both (Sale & Rent)'], weights=[48, 48, 4])[0]

    elif seller_type_code == 'DV':
        seller_name = f"{random.choice(DEVELOPER_NAMES)} Developments"
        allowed_deal_types = 'Sale'

    else:  # 'HS'
        seller_name = f"{random.choice(HOSPITALITY_NAMES)}"
        allowed_deal_types = 'Rent'

    return seller_name, allowed_deal_types

print("✅ Updated seller logic engine (Individual names for brokers + audit dates).")

✅ Updated seller logic engine (Individual names for brokers + audit dates).


#### **3. Synthetic Generation Engine for Dim_Sellers**
###### Generates 200 seller records using structured IDs matching Dim_Properties (OW0000001, DV0000002, etc.).

In [42]:
random.seed(202)  # Seed خاص ببلوك Dim_Sellers (بدل الخلية المحذوفة)

def generate_sellers_dataset(num_records=200):
    records = []

    for i in range(1, num_records + 1):
        type_code = random.choice(['OW', 'BR', 'HS', 'DV'])
        seller_id = f"{type_code}{i:07d}"

        seller_type_label = SELLER_TYPES[type_code]
        seller_name, allowed_deals = get_seller_business_logic(type_code)

        phone = f"+201{random.choice(['0', '1', '2', '5'])}{random.randint(10000000, 99999999)}"
        email_clean = seller_name.lower().replace(' (co-broker)', '').replace(' ', '.').replace('/', '')
        email = f"{email_clean}{random.randint(10,99)}@gmail.com" if type_code in ['OW', 'BR'] else f"contact@{email_clean[:12]}.com"

        # تاريخ ووقت الإدخال الفعلي (بالدقيقة) + الموظف اللي دخّل بيانات البائع
        entry_year = random.choice([2023, 2024, 2025, 2026])
        created_datetime, entered_by = generate_work_datetime(entry_year)

        records.append({
            'SellerID': seller_id,
            'SellerName': seller_name,
            'SellerTypeCode': type_code,
            'SellerType': seller_type_label,
            'OfferingDealType': allowed_deals,
            'Phone': phone,
            'Email': email,
            'PartnerStatus': random.choice(['Active Partner', 'Under Review', 'Verified']),
            'CreatedDateTime': created_datetime.strftime('%Y-%m-%d %H:%M:%S'),
            'EnteredByEmployeeID': entered_by
        })

    return pd.DataFrame(records)

# Run Generation Pipeline
df_dim_sellers = generate_sellers_dataset(num_records=200)
print(f"🎉 Generated {len(df_dim_sellers)} clean records for Dim_Sellers مع تواريخ إدخال دقيقة!")

🎉 Generated 200 clean records for Dim_Sellers مع تواريخ إدخال دقيقة!


#### **4. Sync: Dim_Property , Dim_Sellers**

In [43]:
def resync_property_sellers(df_properties, df_sellers):
    updated_seller_ids = []
    for _, prop in df_properties.iterrows():
        deal = prop['SellingOption']  # Rent / Sale Cash / Sale Installment
        target = 'Rent' if deal == 'Rent' else 'Sale'

        eligible = df_sellers[
            (df_sellers['OfferingDealType'] == target) |
            (df_sellers['OfferingDealType'] == 'Both (Sale & Rent)')
        ]
        if len(eligible) == 0:
            eligible = df_sellers  # حل احتياطي لو مفيش تطابق

        updated_seller_ids.append(random.choice(eligible['SellerID'].tolist()))

    df_properties['SellerID'] = updated_seller_ids
    return df_properties

df_dim_properties = resync_property_sellers(df_dim_properties, df_dim_sellers)
print("✅ SellerID في Dim_Properties بقت 100% متطابقة مع Dim_Sellers، ومتوافقة منطقيًا مع نوع العرض (بيع/إيجار).")

✅ SellerID في Dim_Properties بقت 100% متطابقة مع Dim_Sellers، ومتوافقة منطقيًا مع نوع العرض (بيع/إيجار).


#### **5. Data Validation & Inspection**

In [44]:
print("=== Dim_Sellers Sample Preview ===")
display(df_dim_sellers[['SellerID', 'SellerName', 'SellerType', 'OfferingDealType', 'PartnerStatus', 'CreatedDateTime', 'EnteredByEmployeeID']].head(12))

print("\n=== Offering Deal Type Distribution ===")
print(df_dim_sellers['OfferingDealType'].value_counts())

# فحص إضافي: التأكد إن كل SellerID في Dim_Properties موجود فعليًا في Dim_Sellers
unmatched_sellers = set(df_dim_properties['SellerID']) - set(df_dim_sellers['SellerID'])
print(f"\n🔍 Audit Result: عدد SellerID غير مطابقة بين Properties و Sellers = {len(unmatched_sellers)}")

=== Dim_Sellers Sample Preview ===


,SellerID,SellerName,SellerType,OfferingDealType,PartnerStatus,CreatedDateTime,EnteredByEmployeeID
0,DV0000001,Hassan Allam Developments,Real Estate Developer,Sale,Under Review,2024-04-22 15:03:00,EMP_004
1,DV0000002,Sodic Developments,Real Estate Developer,Sale,Under Review,2024-08-21 12:55:00,EMP_004
2,DV0000003,Mivida / Emaar Developments,Real Estate Developer,Sale,Under Review,2023-05-15 16:55:00,EMP_004
3,OW0000004,Amr El-Ghabour,Individual Owner,Sale,Verified,2024-02-12 11:56:00,EMP_003
4,BR0000005,Rania Mansour (Co-Broker),External Co-Broker,Rent,Verified,2025-08-25 12:22:00,EMP_003
5,BR0000006,Engy El-Shazly (Co-Broker),External Co-Broker,Rent,Active Partner,2024-04-18 14:56:00,EMP_004
6,BR0000007,Ghada El-Ghabour (Co-Broker),External Co-Broker,Rent,Verified,2023-05-24 11:16:00,EMP_004
7,HS0000008,Roft Hotel Apartments,Hospitality Company,Rent,Active Partner,2026-03-15 09:56:00,EMP_003
8,HS0000009,Cairo Luxury Suites,Hospitality Company,Rent,Verified,2024-01-16 10:12:00,EMP_003
9,OW0000010,Amr Sawy,Individual Owner,Sale,Under Review,2023-08-02 13:48:00,EMP_004



=== Offering Deal Type Distribution ===
OfferingDealType
Sale                  102
Rent                   93
Both (Sale & Rent)      5
Name: count, dtype: int64

🔍 Audit Result: عدد SellerID غير مطابقة بين Properties و Sellers = 0


#### **6. Google Sheets Export Pipeline**

In [45]:
FOLDER_NAME = "New_Cairo_RealEstate_DW"
SPREADSHEET_NAME = "NC_RealEstate_DW_Data"
TAB_NAME = "Dim_Sellers"

drive_service = build('drive', 'v3', credentials=creds)

results = drive_service.files().list(
    q=f"name = '{FOLDER_NAME}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
    fields="files(id, name)"
).execute()
folders = results.get('files', [])

if not folders:
    print(f"⚠️ Folder '{FOLDER_NAME}' not found. Please ensure it exists in Google Drive.")
else:
    folder_id = folders[0]['id']

    try:
        sh = gc.open(SPREADSHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(SPREADSHEET_NAME)
        drive_service.files().update(
            fileId=sh.id,
            addParents=folder_id,
            removeParents='root',
            fields='id, parents'
        ).execute()

    try:
        worksheet = sh.worksheet(TAB_NAME)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=TAB_NAME, rows="1000", cols="30")

    worksheet.clear()
    worksheet.update([df_dim_sellers.columns.values.tolist()] + df_dim_sellers.astype(str).values.tolist())

    print(f"🚀 Success! Tab [{TAB_NAME}] exported successfully to [{SPREADSHEET_NAME}] inside '{FOLDER_NAME}' folder.")

🚀 Success! Tab [Dim_Sellers] exported successfully to [NC_RealEstate_DW_Data] inside 'New_Cairo_RealEstate_DW' folder.


## **• Target Table:** Dim_Departments_Roles •

#### **1. Dim_Departments_Roles Data Generation**

In [46]:
import pandas as pd

departments_roles_data = [
    # General Management
    {'DepartmentID': 'DEP_01', 'DepartmentName': 'General Management', 'RoleID': 'ROLE_01', 'RoleName': 'General Manager', 'EstablishedYear': 2023},

    # Operations
    {'DepartmentID': 'DEP_02', 'DepartmentName': 'Operations', 'RoleID': 'ROLE_02', 'RoleName': 'Operations Manager', 'EstablishedYear': 2024},
    {'DepartmentID': 'DEP_02', 'DepartmentName': 'Operations', 'RoleID': 'ROLE_03', 'RoleName': 'Operations Analyst', 'EstablishedYear': 2024},
    {'DepartmentID': 'DEP_02', 'DepartmentName': 'Operations', 'RoleID': 'ROLE_04', 'RoleName': 'Operations Coordinator', 'EstablishedYear': 2024},
    {'DepartmentID': 'DEP_02', 'DepartmentName': 'Operations', 'RoleID': 'ROLE_05', 'RoleName': 'Business Developer', 'EstablishedYear': 2025},

    # Human Resources
    {'DepartmentID': 'DEP_03', 'DepartmentName': 'Human Resources', 'RoleID': 'ROLE_06', 'RoleName': 'HR Senior', 'EstablishedYear': 2024},
    {'DepartmentID': 'DEP_03', 'DepartmentName': 'Human Resources', 'RoleID': 'ROLE_07', 'RoleName': 'HR Junior', 'EstablishedYear': 2025},

    # Accounting
    {'DepartmentID': 'DEP_04', 'DepartmentName': 'Accounting', 'RoleID': 'ROLE_08', 'RoleName': 'Accounting Senior', 'EstablishedYear': 2024},
    {'DepartmentID': 'DEP_04', 'DepartmentName': 'Accounting', 'RoleID': 'ROLE_09', 'RoleName': 'Accounting Junior', 'EstablishedYear': 2025},

    # IT
    {'DepartmentID': 'DEP_05', 'DepartmentName': 'IT', 'RoleID': 'ROLE_10', 'RoleName': 'IT Senior', 'EstablishedYear': 2024},
    {'DepartmentID': 'DEP_05', 'DepartmentName': 'IT', 'RoleID': 'ROLE_11', 'RoleName': 'IT Junior', 'EstablishedYear': 2025},

    # Sales
    {'DepartmentID': 'DEP_06', 'DepartmentName': 'Sales', 'RoleID': 'ROLE_12', 'RoleName': 'Sales Team Leader', 'EstablishedYear': 2024},
    {'DepartmentID': 'DEP_06', 'DepartmentName': 'Sales', 'RoleID': 'ROLE_13', 'RoleName': 'Sales Senior', 'EstablishedYear': 2024},
    {'DepartmentID': 'DEP_06', 'DepartmentName': 'Sales', 'RoleID': 'ROLE_14', 'RoleName': 'Sales Junior', 'EstablishedYear': 2023},

    # Data Management
    {'DepartmentID': 'DEP_07', 'DepartmentName': 'Data Management', 'RoleID': 'ROLE_15', 'RoleName': 'Sales Admin Senior', 'EstablishedYear': 2024},
    {'DepartmentID': 'DEP_07', 'DepartmentName': 'Data Management', 'RoleID': 'ROLE_16', 'RoleName': 'Sales Admin Junior', 'EstablishedYear': 2023},
    {'DepartmentID': 'DEP_07', 'DepartmentName': 'Data Management', 'RoleID': 'ROLE_17', 'RoleName': 'Moderator', 'EstablishedYear': 2023},
    {'DepartmentID': 'DEP_07', 'DepartmentName': 'Data Management', 'RoleID': 'ROLE_18', 'RoleName': 'Data Collector', 'EstablishedYear': 2023},

    # Marketing
    {'DepartmentID': 'DEP_08', 'DepartmentName': 'Marketing', 'RoleID': 'ROLE_19', 'RoleName': 'Marketing Senior', 'EstablishedYear': 2025},
    {'DepartmentID': 'DEP_08', 'DepartmentName': 'Marketing', 'RoleID': 'ROLE_20', 'RoleName': 'Marketing Junior', 'EstablishedYear': 2024}
]

df_dim_departments_roles = pd.DataFrame(departments_roles_data)

print("=== Dim_Departments_Roles Data Preview ===")
display(df_dim_departments_roles.head(10))
print(f"✅ Total Roles Configured: {len(df_dim_departments_roles)}")

=== Dim_Departments_Roles Data Preview ===


,DepartmentID,DepartmentName,RoleID,RoleName,EstablishedYear
0,DEP_01,General Management,ROLE_01,General Manager,2023
1,DEP_02,Operations,ROLE_02,Operations Manager,2024
2,DEP_02,Operations,ROLE_03,Operations Analyst,2024
3,DEP_02,Operations,ROLE_04,Operations Coordinator,2024
4,DEP_02,Operations,ROLE_05,Business Developer,2025
5,DEP_03,Human Resources,ROLE_06,HR Senior,2024
6,DEP_03,Human Resources,ROLE_07,HR Junior,2025
7,DEP_04,Accounting,ROLE_08,Accounting Senior,2024
8,DEP_04,Accounting,ROLE_09,Accounting Junior,2025
9,DEP_05,IT,ROLE_10,IT Senior,2024


✅ Total Roles Configured: 20


#### **2. Data Validation & Inspection**

In [47]:
duplicate_roles = df_dim_departments_roles[df_dim_departments_roles.duplicated('RoleID')]
print(f"🔍 Audit Result: عدد RoleID مكرر = {len(duplicate_roles)}")

🔍 Audit Result: عدد RoleID مكرر = 0


#### **3. Live Google Sheets Export**

In [48]:
TAB_NAME = "Dim_Departments_Roles"

if folders:
    try:
        sh = gc.open(SPREADSHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(SPREADSHEET_NAME)
        drive_service.files().update(
            fileId=sh.id,
            addParents=folder_id,
            removeParents='root',
            fields='id, parents'
        ).execute()

    try:
        worksheet = sh.worksheet(TAB_NAME)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=TAB_NAME, rows="100", cols="10")

    worksheet.clear()
    worksheet.update([df_dim_departments_roles.columns.values.tolist()] + df_dim_departments_roles.astype(str).values.tolist())

    print(f"🚀 Success! [{len(df_dim_departments_roles)}] records synchronized to tab [{TAB_NAME}] in Google Sheets.")

🚀 Success! [20] records synchronized to tab [Dim_Departments_Roles] in Google Sheets.


## **• Target Tables:** Dim_Employees & Dim_Employee_Career_History •

#### **1. Dim_Employees**

In [49]:
import pandas as pd

employees_raw_data = [
    # --- 2023 Bootstrap Founders ---
    {
        'EmployeeID': 'EMP_001', 'FullName': 'Ahmed El-Sayed', 'Gender': 'Male',
        'DepartmentID': 'DEP_01', 'CurrentRoleID': 'ROLE_01', 'HireDate': '2023-01-01', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 45000, 'CommissionRate': 0.0,
        'CareerHistory': [
            {'EffectiveDate': '2023-01-01', 'RoleID': 'ROLE_01', 'RoleName': 'General Manager', 'Salary': 45000, 'EventType': 'Hire'}
        ]
    },
    {
        'EmployeeID': 'EMP_002', 'FullName': 'Mahmoud Hassan', 'Gender': 'Male',
        'DepartmentID': 'DEP_06', 'CurrentRoleID': 'ROLE_12', 'HireDate': '2023-01-15', 'ExitDate': None,
        'Status': 'Promoted', 'BaseSalary_EGP': 8000, 'CommissionRate': 0.20,
        'CareerHistory': [
            {'EffectiveDate': '2023-01-15', 'RoleID': 'ROLE_14', 'RoleName': 'Sales Junior', 'Salary': 2000, 'EventType': 'Hire'},
            {'EffectiveDate': '2024-06-01', 'RoleID': 'ROLE_13', 'RoleName': 'Sales Senior', 'Salary': 4500, 'EventType': 'Promotion'},
            {'EffectiveDate': '2025-08-01', 'RoleID': 'ROLE_12', 'RoleName': 'Sales Team Leader', 'Salary': 8000, 'EventType': 'Promotion'}
        ]
    },
    {
        'EmployeeID': 'EMP_003', 'FullName': 'Nouran Ali', 'Gender': 'Female',
        'DepartmentID': 'DEP_07', 'CurrentRoleID': 'ROLE_15', 'HireDate': '2023-01-15', 'ExitDate': None,
        'Status': 'Promoted', 'BaseSalary_EGP': 14000, 'CommissionRate': 0.01,
        'CareerHistory': [
            {'EffectiveDate': '2023-01-15', 'RoleID': 'ROLE_16', 'RoleName': 'Sales Admin Junior', 'Salary': 5000, 'EventType': 'Hire'},
            {'EffectiveDate': '2024-09-01', 'RoleID': 'ROLE_15', 'RoleName': 'Sales Admin Senior', 'Salary': 14000, 'EventType': 'Promotion'}
        ]
    },
    {
        'EmployeeID': 'EMP_004', 'FullName': 'Kareem Fahmy', 'Gender': 'Male',
        'DepartmentID': 'DEP_07', 'CurrentRoleID': 'ROLE_17', 'HireDate': '2023-02-01', 'ExitDate': None,
        'Status': 'Promoted', 'BaseSalary_EGP': 9500, 'CommissionRate': 0.005,
        'CareerHistory': [
            {'EffectiveDate': '2023-02-01', 'RoleID': 'ROLE_18', 'RoleName': 'Data Collector / Moderator', 'Salary': 4000, 'EventType': 'Hire'},
            {'EffectiveDate': '2024-05-01', 'RoleID': 'ROLE_17', 'RoleName': 'Moderator', 'Salary': 9500, 'EventType': 'Promotion'}
        ]
    },

    # --- 2024 Expansion & Turnover Events ---
    {
        'EmployeeID': 'EMP_005', 'FullName': 'Omar Khaled', 'Gender': 'Male',
        'DepartmentID': 'DEP_02', 'CurrentRoleID': 'ROLE_02', 'HireDate': '2024-01-10', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 28000, 'CommissionRate': 0.0,
        'CareerHistory': [{'EffectiveDate': '2024-01-10', 'RoleID': 'ROLE_02', 'RoleName': 'Operations Manager', 'Salary': 28000, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_006', 'FullName': 'Sherif Abdelaziz', 'Gender': 'Male',
        'DepartmentID': 'DEP_02', 'CurrentRoleID': 'ROLE_03', 'HireDate': '2024-03-01', 'ExitDate': '2024-11-15',
        'Status': 'Resigned', 'BaseSalary_EGP': 15000, 'CommissionRate': 0.0,
        'CareerHistory': [
            {'EffectiveDate': '2024-03-01', 'RoleID': 'ROLE_03', 'RoleName': 'Operations Analyst', 'Salary': 15000, 'EventType': 'Hire'},
            {'EffectiveDate': '2024-11-15', 'RoleID': 'ROLE_03', 'RoleName': 'Operations Analyst', 'Salary': 15000, 'EventType': 'Resignation'}
        ]
    },
    {
        'EmployeeID': 'EMP_007', 'FullName': 'Dina Reda', 'Gender': 'Female',
        'DepartmentID': 'DEP_03', 'CurrentRoleID': 'ROLE_06', 'HireDate': '2024-02-01', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 18000, 'CommissionRate': 0.0,
        'CareerHistory': [{'EffectiveDate': '2024-02-01', 'RoleID': 'ROLE_06', 'RoleName': 'HR Senior', 'Salary': 18000, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_008', 'FullName': 'Tarek Mostafa', 'Gender': 'Male',
        'DepartmentID': 'DEP_04', 'CurrentRoleID': 'ROLE_08', 'HireDate': '2024-01-15', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 20000, 'CommissionRate': 0.0,
        'CareerHistory': [{'EffectiveDate': '2024-01-15', 'RoleID': 'ROLE_08', 'RoleName': 'Accounting Senior', 'Salary': 20000, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_009', 'FullName': 'Youssef Gamal', 'Gender': 'Male',
        'DepartmentID': 'DEP_05', 'CurrentRoleID': 'ROLE_10', 'HireDate': '2024-02-15', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 22000, 'CommissionRate': 0.0,
        'CareerHistory': [{'EffectiveDate': '2024-02-15', 'RoleID': 'ROLE_10', 'RoleName': 'IT Senior', 'Salary': 22000, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_010', 'FullName': 'Sama Ibrahim', 'Gender': 'Female',
        'DepartmentID': 'DEP_08', 'CurrentRoleID': 'ROLE_19', 'HireDate': '2024-04-01', 'ExitDate': None,
        'Status': 'Promoted', 'BaseSalary_EGP': 16000, 'CommissionRate': 0.0,
        'CareerHistory': [
            {'EffectiveDate': '2024-04-01', 'RoleID': 'ROLE_20', 'RoleName': 'Marketing Junior', 'Salary': 9000, 'EventType': 'Hire'},
            {'EffectiveDate': '2025-05-01', 'RoleID': 'ROLE_19', 'RoleName': 'Marketing Senior', 'Salary': 16000, 'EventType': 'Promotion'}
        ]
    },

    # --- Sales Department Turnover & Promotions ---
    {
        'EmployeeID': 'EMP_011', 'FullName': 'Mostafa Mahmoud', 'Gender': 'Male',
        'DepartmentID': 'DEP_06', 'CurrentRoleID': 'ROLE_13', 'HireDate': '2024-02-01', 'ExitDate': None,
        'Status': 'Promoted', 'BaseSalary_EGP': 4500, 'CommissionRate': 0.20,
        'CareerHistory': [
            {'EffectiveDate': '2024-02-01', 'RoleID': 'ROLE_14', 'RoleName': 'Sales Junior', 'Salary': 2500, 'EventType': 'Hire'},
            {'EffectiveDate': '2025-06-01', 'RoleID': 'ROLE_13', 'RoleName': 'Sales Senior', 'Salary': 4500, 'EventType': 'Promotion'}
        ]
    },
    {
        'EmployeeID': 'EMP_012', 'FullName': 'Hassan Hamdy', 'Gender': 'Male',
        'DepartmentID': 'DEP_06', 'CurrentRoleID': 'ROLE_14', 'HireDate': '2024-03-15', 'ExitDate': '2025-02-28',
        'Status': 'Resigned', 'BaseSalary_EGP': 2500, 'CommissionRate': 0.20,
        'CareerHistory': [
            {'EffectiveDate': '2024-03-15', 'RoleID': 'ROLE_14', 'RoleName': 'Sales Junior', 'Salary': 2500, 'EventType': 'Hire'},
            {'EffectiveDate': '2025-02-28', 'RoleID': 'ROLE_14', 'RoleName': 'Sales Junior', 'Salary': 2500, 'EventType': 'Resignation'}
        ]
    },
    {
        'EmployeeID': 'EMP_013', 'FullName': 'Sarah Nabil', 'Gender': 'Female',
        'DepartmentID': 'DEP_06', 'CurrentRoleID': 'ROLE_13', 'HireDate': '2024-05-01', 'ExitDate': None,
        'Status': 'Promoted', 'BaseSalary_EGP': 5000, 'CommissionRate': 0.20,
        'CareerHistory': [
            {'EffectiveDate': '2024-05-01', 'RoleID': 'ROLE_14', 'RoleName': 'Sales Junior', 'Salary': 2500, 'EventType': 'Hire'},
            {'EffectiveDate': '2025-09-01', 'RoleID': 'ROLE_13', 'RoleName': 'Sales Senior', 'Salary': 5000, 'EventType': 'Promotion'}
        ]
    },
    {
        'EmployeeID': 'EMP_014', 'FullName': 'Eslam Samy', 'Gender': 'Male',
        'DepartmentID': 'DEP_06', 'CurrentRoleID': 'ROLE_14', 'HireDate': '2024-07-01', 'ExitDate': '2024-12-31',
        'Status': 'Terminated', 'BaseSalary_EGP': 2500, 'CommissionRate': 0.20,
        'CareerHistory': [
            {'EffectiveDate': '2024-07-01', 'RoleID': 'ROLE_14', 'RoleName': 'Sales Junior', 'Salary': 2500, 'EventType': 'Hire'},
            {'EffectiveDate': '2024-12-31', 'RoleID': 'ROLE_14', 'RoleName': 'Sales Junior', 'Salary': 2500, 'EventType': 'Termination'}
        ]
    },
    {
        'EmployeeID': 'EMP_015', 'FullName': 'Amr Zaki', 'Gender': 'Male',
        'DepartmentID': 'DEP_06', 'CurrentRoleID': 'ROLE_13', 'HireDate': '2024-09-01', 'ExitDate': None,
        'Status': 'Promoted', 'BaseSalary_EGP': 4500, 'CommissionRate': 0.20,
        'CareerHistory': [
            {'EffectiveDate': '2024-09-01', 'RoleID': 'ROLE_14', 'RoleName': 'Sales Junior', 'Salary': 2500, 'EventType': 'Hire'},
            {'EffectiveDate': '2026-01-15', 'RoleID': 'ROLE_13', 'RoleName': 'Sales Senior', 'Salary': 4500, 'EventType': 'Promotion'}
        ]
    },

    # --- 2025 - 2026 Expansion & Additional Terminations/Resignations ---
    {
        'EmployeeID': 'EMP_016', 'FullName': 'Mona Farouk', 'Gender': 'Female',
        'DepartmentID': 'DEP_08', 'CurrentRoleID': 'ROLE_20', 'HireDate': '2025-05-15', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 9500, 'CommissionRate': 0.0,
        'CareerHistory': [{'EffectiveDate': '2025-05-15', 'RoleID': 'ROLE_20', 'RoleName': 'Marketing Junior', 'Salary': 9500, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_017', 'FullName': 'Hady Badr', 'Gender': 'Male',
        'DepartmentID': 'DEP_06', 'CurrentRoleID': 'ROLE_14', 'HireDate': '2025-01-10', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 3000, 'CommissionRate': 0.20,
        'CareerHistory': [{'EffectiveDate': '2025-01-10', 'RoleID': 'ROLE_14', 'RoleName': 'Sales Junior', 'Salary': 3000, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_018', 'FullName': 'Yasmin Sherif', 'Gender': 'Female',
        'DepartmentID': 'DEP_06', 'CurrentRoleID': 'ROLE_14', 'HireDate': '2025-03-01', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 3000, 'CommissionRate': 0.20,
        'CareerHistory': [{'EffectiveDate': '2025-03-01', 'RoleID': 'ROLE_14', 'RoleName': 'Sales Junior', 'Salary': 3000, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_019', 'FullName': 'Nader Shaker', 'Gender': 'Male',
        'DepartmentID': 'DEP_06', 'CurrentRoleID': 'ROLE_14', 'HireDate': '2025-06-01', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 3000, 'CommissionRate': 0.20,
        'CareerHistory': [{'EffectiveDate': '2025-06-01', 'RoleID': 'ROLE_14', 'RoleName': 'Sales Junior', 'Salary': 3000, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_020', 'FullName': 'Mariam Hossam', 'Gender': 'Female',
        'DepartmentID': 'DEP_07', 'CurrentRoleID': 'ROLE_16', 'HireDate': '2025-02-01', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 6500, 'CommissionRate': 0.005,
        'CareerHistory': [{'EffectiveDate': '2025-02-01', 'RoleID': 'ROLE_16', 'RoleName': 'Sales Admin Junior', 'Salary': 6500, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_021', 'FullName': 'Ramy Wagdy', 'Gender': 'Male',
        'DepartmentID': 'DEP_07', 'CurrentRoleID': 'ROLE_18', 'HireDate': '2025-04-01', 'ExitDate': '2025-10-31',
        'Status': 'Terminated', 'BaseSalary_EGP': 6000, 'CommissionRate': 0.0005,
        'CareerHistory': [
            {'EffectiveDate': '2025-04-01', 'RoleID': 'ROLE_18', 'RoleName': 'Data Collector', 'Salary': 6000, 'EventType': 'Hire'},
            {'EffectiveDate': '2025-10-31', 'RoleID': 'ROLE_18', 'RoleName': 'Data Collector', 'Salary': 6000, 'EventType': 'Termination'}
        ]
    },
    {
        'EmployeeID': 'EMP_022', 'FullName': 'Nada Hazem', 'Gender': 'Female',
        'DepartmentID': 'DEP_03', 'CurrentRoleID': 'ROLE_07', 'HireDate': '2025-07-01', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 8500, 'CommissionRate': 0.0,
        'CareerHistory': [{'EffectiveDate': '2025-07-01', 'RoleID': 'ROLE_07', 'RoleName': 'HR Junior', 'Salary': 8500, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_023', 'FullName': 'Wael Lotfy', 'Gender': 'Male',
        'DepartmentID': 'DEP_04', 'CurrentRoleID': 'ROLE_09', 'HireDate': '2025-08-01', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 9000, 'CommissionRate': 0.0,
        'CareerHistory': [{'EffectiveDate': '2025-08-01', 'RoleID': 'ROLE_09', 'RoleName': 'Accounting Junior', 'Salary': 9000, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_024', 'FullName': 'Ziad Sameh', 'Gender': 'Male',
        'DepartmentID': 'DEP_05', 'CurrentRoleID': 'ROLE_11', 'HireDate': '2025-09-01', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 10000, 'CommissionRate': 0.0,
        'CareerHistory': [{'EffectiveDate': '2025-09-01', 'RoleID': 'ROLE_11', 'RoleName': 'IT Junior', 'Salary': 10000, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_025', 'FullName': 'Khaled Souka', 'Gender': 'Male',
        'DepartmentID': 'DEP_02', 'CurrentRoleID': 'ROLE_04', 'HireDate': '2025-03-15', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 8500, 'CommissionRate': 0.0,
        'CareerHistory': [{'EffectiveDate': '2025-03-15', 'RoleID': 'ROLE_04', 'RoleName': 'Operations Coordinator', 'Salary': 8500, 'EventType': 'Hire'}]
    },
    {
        'EmployeeID': 'EMP_026', 'FullName': 'Adham Metwally', 'Gender': 'Male',
        'DepartmentID': 'DEP_02', 'CurrentRoleID': 'ROLE_05', 'HireDate': '2026-01-10', 'ExitDate': None,
        'Status': 'Active', 'BaseSalary_EGP': 12000, 'CommissionRate': 0.0,
        'CareerHistory': [{'EffectiveDate': '2026-01-10', 'RoleID': 'ROLE_05', 'RoleName': 'Business Developer', 'Salary': 12000, 'EventType': 'Hire'}]
    }
]


MANAGER_MAP = {
    'EMP_001': '',            # General Manager - قمة الهيكل
    'EMP_002': 'EMP_001', 'EMP_003': 'EMP_001', 'EMP_005': 'EMP_001',
    'EMP_007': 'EMP_001', 'EMP_008': 'EMP_001', 'EMP_009': 'EMP_001', 'EMP_010': 'EMP_001',
    'EMP_004': 'EMP_003', 'EMP_020': 'EMP_003', 'EMP_021': 'EMP_004',
    'EMP_006': 'EMP_005', 'EMP_025': 'EMP_005', 'EMP_026': 'EMP_005',
    'EMP_011': 'EMP_002', 'EMP_012': 'EMP_002', 'EMP_013': 'EMP_002', 'EMP_014': 'EMP_002',
    'EMP_015': 'EMP_002', 'EMP_017': 'EMP_002', 'EMP_018': 'EMP_002', 'EMP_019': 'EMP_002',
    'EMP_016': 'EMP_010', 'EMP_022': 'EMP_007', 'EMP_023': 'EMP_008', 'EMP_024': 'EMP_009'
}

#### **2. Career History Generation - DataFrame**

In [50]:
employees_list = []
career_history_list = []
history_counter = 1

for emp in employees_raw_data:
    employees_list.append({
        'EmployeeID': emp['EmployeeID'],
        'FullName': emp['FullName'],
        'Gender': emp['Gender'],
        'DepartmentID': emp['DepartmentID'],
        'CurrentRoleID': emp['CurrentRoleID'],
        'HireDate': emp['HireDate'],
        'ExitDate': emp['ExitDate'] if emp['ExitDate'] else '',   # فاضي بدل 'N/A' لأن SQL يقبلها NULL
        'Status': emp['Status'],
        'BaseSalary_EGP': emp['BaseSalary_EGP'],
        'CommissionRate': emp['CommissionRate'],
        'ManagerID': MANAGER_MAP.get(emp['EmployeeID'], '')
    })

    for history in emp['CareerHistory']:
        career_history_list.append({
            'HistoryID': f"HIST_{history_counter:04d}",
            'EmployeeID': emp['EmployeeID'],
            'EffectiveDate': history['EffectiveDate'],
            'RoleID': history['RoleID'],
            'RoleName': history['RoleName'],
            'Salary_EGP': history['Salary'],
            'EventType': history['EventType']
        })
        history_counter += 1

df_dim_employees = pd.DataFrame(employees_list)
df_dim_employee_career_history = pd.DataFrame(career_history_list)


#### **3. Case Statistics on the Dashboard**

In [51]:
status_counts = df_dim_employees['Status'].value_counts()

print("=== Dim_Employees Status Breakdown ===")
print(status_counts)

print("\n=== Dim_Employees Sample Data ===")
display(df_dim_employees.head(10))

print("=== Dim_Employee_Career_History Sample Data ===")
display(df_dim_employee_career_history.head(10))

=== Dim_Employees Status Breakdown ===
Status
Active        15
Promoted       7
Resigned       2
Terminated     2
Name: count, dtype: int64

=== Dim_Employees Sample Data ===


,EmployeeID,FullName,Gender,DepartmentID,CurrentRoleID,HireDate,ExitDate,Status,BaseSalary_EGP,CommissionRate,ManagerID
0,EMP_001,Ahmed El-Sayed,Male,DEP_01,ROLE_01,2023-01-01,,Active,45000,0.000,
1,EMP_002,Mahmoud Hassan,Male,DEP_06,ROLE_12,2023-01-15,,Promoted,8000,0.200,EMP_001
2,EMP_003,Nouran Ali,Female,DEP_07,ROLE_15,2023-01-15,,Promoted,14000,0.010,EMP_001
3,EMP_004,Kareem Fahmy,Male,DEP_07,ROLE_17,2023-02-01,,Promoted,9500,0.005,EMP_003
4,EMP_005,Omar Khaled,Male,DEP_02,ROLE_02,2024-01-10,,Active,28000,0.000,EMP_001
5,EMP_006,Sherif Abdelaziz,Male,DEP_02,ROLE_03,2024-03-01,2024-11-15,Resigned,15000,0.000,EMP_005
6,EMP_007,Dina Reda,Female,DEP_03,ROLE_06,2024-02-01,,Active,18000,0.000,EMP_001
7,EMP_008,Tarek Mostafa,Male,DEP_04,ROLE_08,2024-01-15,,Active,20000,0.000,EMP_001
8,EMP_009,Youssef Gamal,Male,DEP_05,ROLE_10,2024-02-15,,Active,22000,0.000,EMP_001
9,EMP_010,Sama Ibrahim,Female,DEP_08,ROLE_19,2024-04-01,,Promoted,16000,0.000,EMP_001


=== Dim_Employee_Career_History Sample Data ===


,HistoryID,EmployeeID,EffectiveDate,RoleID,RoleName,Salary_EGP,EventType
0,HIST_0001,EMP_001,2023-01-01,ROLE_01,General Manager,45000,Hire
1,HIST_0002,EMP_002,2023-01-15,ROLE_14,Sales Junior,2000,Hire
2,HIST_0003,EMP_002,2024-06-01,ROLE_13,Sales Senior,4500,Promotion
3,HIST_0004,EMP_002,2025-08-01,ROLE_12,Sales Team Leader,8000,Promotion
4,HIST_0005,EMP_003,2023-01-15,ROLE_16,Sales Admin Junior,5000,Hire
5,HIST_0006,EMP_003,2024-09-01,ROLE_15,Sales Admin Senior,14000,Promotion
6,HIST_0007,EMP_004,2023-02-01,ROLE_18,Data Collector / Moderator,4000,Hire
7,HIST_0008,EMP_004,2024-05-01,ROLE_17,Moderator,9500,Promotion
8,HIST_0009,EMP_005,2024-01-10,ROLE_02,Operations Manager,28000,Hire
9,HIST_0010,EMP_006,2024-03-01,ROLE_03,Operations Analyst,15000,Hire



#### **4. Integrity check:**
######Every ManagerID (if present) must be a valid EmployeeID existing in the same table.

In [52]:
valid_ids = set(df_dim_employees['EmployeeID'])
invalid_managers = df_dim_employees[
    (df_dim_employees['ManagerID'] != '') & (~df_dim_employees['ManagerID'].isin(valid_ids))
]
print(f"🔍 Audit Result: عدد ManagerID غير صالح = {len(invalid_managers)}")

🔍 Audit Result: عدد ManagerID غير صالح = 0


### **5. Live Google Sheets Export**

In [53]:
TAB_NAME_EMP = "Dim_Employees"
TAB_NAME_HIST = "Dim_Employee_Career_History"

if folders:

    try:
        sh = gc.open(SPREADSHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(SPREADSHEET_NAME)
        drive_service.files().update(
            fileId=sh.id,
            addParents=folder_id,
            removeParents='root',
            fields='id, parents'
        ).execute()

    # --- تصدير Dim_Employees ---
    try:
        worksheet_emp = sh.worksheet(TAB_NAME_EMP)
    except gspread.exceptions.WorksheetNotFound:
        worksheet_emp = sh.add_worksheet(title=TAB_NAME_EMP, rows="200", cols="10")

    worksheet_emp.clear()
    worksheet_emp.update([df_dim_employees.columns.values.tolist()] + df_dim_employees.astype(str).values.tolist())
    print(f"🚀 [{TAB_NAME_EMP}] exported successfully.")

    # --- تصدير Dim_Employee_Career_History ---
    try:
        worksheet_hist = sh.worksheet(TAB_NAME_HIST)
    except gspread.exceptions.WorksheetNotFound:
        worksheet_hist = sh.add_worksheet(title=TAB_NAME_HIST, rows="200", cols="10")

    worksheet_hist.clear()
    worksheet_hist.update([df_dim_employee_career_history.columns.values.tolist()] + df_dim_employee_career_history.astype(str).values.tolist())
    print(f"🚀 [{TAB_NAME_HIST}] exported successfully.")

🚀 [Dim_Employees] exported successfully.
🚀 [Dim_Employee_Career_History] exported successfully.


## **• Target Table:** Dim_Marketing_Leads •


#### **1. Dim_Marketing_Leads Data Generation**

In [54]:
import pandas as pd
import numpy as np

marketing_campaigns_setup = [
    {'CampaignID': 'CMP_2023_ORG', 'CampaignName': 'Organic Search & Direct', 'Platform': 'Organic / Direct', 'Budget_EGP': 0, 'LaunchYear': 2023},
    {'CampaignID': 'CMP_2023_REF', 'CampaignName': 'Client Referral Program', 'Platform': 'Referral', 'Budget_EGP': 15000, 'LaunchYear': 2023},
    {'CampaignID': 'CMP_2024_FB01', 'CampaignName': 'New Cairo Launch - FB', 'Platform': 'Facebook Ads', 'Budget_EGP': 120000, 'LaunchYear': 2024},
    {'CampaignID': 'CMP_2024_GO01', 'CampaignName': 'Luxury Compound Search - Google', 'Platform': 'Google Ads', 'Budget_EGP': 150000, 'LaunchYear': 2024},
    {'CampaignID': 'CMP_2024_WA01', 'CampaignName': 'Direct WhatsApp Outreach', 'Platform': 'WhatsApp Marketing', 'Budget_EGP': 30000, 'LaunchYear': 2024},
    {'CampaignID': 'CMP_2025_FB02', 'CampaignName': 'Fifth Settlement Villa Ads', 'Platform': 'Facebook Ads', 'Budget_EGP': 200000, 'LaunchYear': 2025},
    {'CampaignID': 'CMP_2025_IG01', 'CampaignName': 'Instagram Visual Showcase', 'Platform': 'Instagram Ads', 'Budget_EGP': 180000, 'LaunchYear': 2025},
    {'CampaignID': 'CMP_2025_GO02', 'CampaignName': 'Commercial Property Search', 'Platform': 'Google Ads', 'Budget_EGP': 220000, 'LaunchYear': 2025},
    {'CampaignID': 'CMP_2026_FB03', 'CampaignName': 'Golden Square Apartments', 'Platform': 'Facebook Ads', 'Budget_EGP': 150000, 'LaunchYear': 2026},
    {'CampaignID': 'CMP_2026_IG02', 'CampaignName': 'Rehab & Madinaty Retargeting', 'Platform': 'Instagram Ads', 'Budget_EGP': 110000, 'LaunchYear': 2026}
]

df_dim_marketing_campaigns = pd.DataFrame(marketing_campaigns_setup)

PROJECT_END_DATE = datetime(2026, 5, 1)

def generate_lead_received_datetime(launch_year):
    campaign_start = datetime(launch_year, 1, 1)
    total_seconds = int((PROJECT_END_DATE - campaign_start).total_seconds())
    random_offset = random.randint(0, max(total_seconds, 1))
    return campaign_start + timedelta(seconds=random_offset)

np.random.seed(42)
total_leads = 1200

lead_ids = [f"LEAD_{str(i).zfill(5)}" for i in range(1, total_leads + 1)]
campaign_choices = np.random.choice(
    df_dim_marketing_campaigns['CampaignID'],
    size=total_leads,
    p=[0.10, 0.08, 0.15, 0.15, 0.07, 0.12, 0.11, 0.10, 0.07, 0.05]
)

# 🆕 مجموعة الـ ClientID الحقيقية من Dim_Clients (هيتم السحب منها لكل Lead)
client_ids_pool = df_dim_clients['ClientID'].tolist()

leads_data = []
for i in range(total_leads):
    cmp_id = campaign_choices[i]
    cmp_info = df_dim_marketing_campaigns[df_dim_marketing_campaigns['CampaignID'] == cmp_id].iloc[0]

    received_datetime = generate_lead_received_datetime(cmp_info['LaunchYear'])

    leads_data.append({
        'LeadID': lead_ids[i],
        'ClientID': random.choice(client_ids_pool),   # 🆕 ربط حقيقي بعميل من Dim_Clients (بيسمح بالتكرار = عميل راجع)
        'CampaignID': cmp_id,
        'CampaignName': cmp_info['CampaignName'],
        'Platform': cmp_info['Platform'],
        'LeadQualityScore': np.random.choice(['Hot', 'Warm', 'Cold'], p=[0.25, 0.50, 0.25]),
        'ReceivedDateTime': received_datetime.strftime('%Y-%m-%d %H:%M:%S')
        # 🗑️ اتشالت CampaignBudget_EGP من هنا - هتيجي من Dim_Marketing_Campaigns بدل ما تتكرر غلط
    })

df_dim_marketing_leads = pd.DataFrame(leads_data)

print("=== Dim_Marketing_Leads Sample Preview ===")
display(df_dim_marketing_leads.head(10))
print(f"🎉 Generated {len(df_dim_marketing_leads)} leads across {len(df_dim_marketing_campaigns)} campaigns!")

# فحص سريع: عدد العملاء اللي ظهروا في أكتر من Lead (يعني عندهم فرصة "Returning")
repeat_clients = df_dim_marketing_leads['ClientID'].value_counts()
print(f"\n🔍 عدد العملاء اللي ظهروا في أكتر من Lead واحد = {(repeat_clients > 1).sum()}")

=== Dim_Marketing_Leads Sample Preview ===


,LeadID,ClientID,CampaignID,CampaignName,Platform,LeadQualityScore,ReceivedDateTime
0,LEAD_00001,CL_0000002,CMP_2024_GO01,Luxury Compound Search - Google,Google Ads,Cold,2025-12-05 03:53:00
1,LEAD_00002,CL_0000165,CMP_2026_IG02,Rehab & Madinaty Retargeting,Instagram Ads,Hot,2026-03-29 00:53:21
2,LEAD_00003,CL_0000298,CMP_2025_IG01,Instagram Visual Showcase,Instagram Ads,Hot,2025-02-14 21:20:56
3,LEAD_00004,CL_0000342,CMP_2025_FB02,Fifth Settlement Villa Ads,Facebook Ads,Warm,2025-05-07 06:02:43
4,LEAD_00005,CL_0000212,CMP_2023_REF,Client Referral Program,Referral,Warm,2025-05-29 14:57:46
5,LEAD_00006,CL_0000344,CMP_2023_REF,Client Referral Program,Referral,Cold,2024-07-31 22:31:26
6,LEAD_00007,CL_0000380,CMP_2023_ORG,Organic Search & Direct,Organic / Direct,Warm,2023-01-22 08:33:22
7,LEAD_00008,CL_0000391,CMP_2025_GO02,Commercial Property Search,Google Ads,Warm,2025-05-23 20:08:41
8,LEAD_00009,CL_0000099,CMP_2025_FB02,Fifth Settlement Villa Ads,Facebook Ads,Warm,2025-06-21 21:01:58
9,LEAD_00010,CL_0000053,CMP_2025_IG01,Instagram Visual Showcase,Instagram Ads,Cold,2025-01-04 14:53:00


🎉 Generated 1200 leads across 10 campaigns!

🔍 عدد العملاء اللي ظهروا في أكتر من Lead واحد = 337


#### **2. Integrity check:**
###### LeadID & CampaignID

In [55]:
duplicate_leads = df_dim_marketing_leads[df_dim_marketing_leads.duplicated('LeadID')]
unmatched_campaigns = set(df_dim_marketing_leads['CampaignID']) - set(df_dim_marketing_campaigns['CampaignID'])

print(f"🔍 Audit: LeadID مكرر = {len(duplicate_leads)}")
print(f"🔍 Audit: CampaignID غير مطابق = {len(unmatched_campaigns)}")
print("\n=== Lead Distribution by Quality ===")
print(df_dim_marketing_leads['LeadQualityScore'].value_counts())

🔍 Audit: LeadID مكرر = 0
🔍 Audit: CampaignID غير مطابق = 0

=== Lead Distribution by Quality ===
LeadQualityScore
Warm    605
Hot     305
Cold    290
Name: count, dtype: int64


#### **3. Live Google Sheets Export**
###### Dim_Marketing_Lead

In [56]:
TAB_NAME_LEADS = "Dim_Marketing_Leads"

if folders:
    try:
        sh = gc.open(SPREADSHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(SPREADSHEET_NAME)
        drive_service.files().update(
            fileId=sh.id,
            addParents=folder_id,
            removeParents='root',
            fields='id, parents'
        ).execute()

    try:
        worksheet = sh.worksheet(TAB_NAME_LEADS)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=TAB_NAME_LEADS, rows="1500", cols="10")

    worksheet.clear()
    worksheet.update([df_dim_marketing_leads.columns.values.tolist()] + df_dim_marketing_leads.astype(str).values.tolist())

    print(f"🚀 Success! [{TAB_NAME_LEADS}] exported successfully to [{SPREADSHEET_NAME}].")

🚀 Success! [Dim_Marketing_Leads] exported successfully to [NC_RealEstate_DW_Data].


#### **3. Live Google Sheets Export**
###### Dim_Marketing_Campaigns

In [57]:
TAB_NAME_CAMPAIGNS = "Dim_Marketing_Campaigns"

if folders:
    try:
        sh = gc.open(SPREADSHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(SPREADSHEET_NAME)

    try:
        worksheet = sh.worksheet(TAB_NAME_CAMPAIGNS)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=TAB_NAME_CAMPAIGNS, rows="20", cols="10")

    worksheet.clear()
    worksheet.update([df_dim_marketing_campaigns.columns.values.tolist()] + df_dim_marketing_campaigns.astype(str).values.tolist())

    print(f"🚀 Success! [{TAB_NAME_CAMPAIGNS}] exported successfully.")

🚀 Success! [Dim_Marketing_Campaigns] exported successfully.


## **• Target Table:** Fact_Leads_Operations •

#### **1. Fact_Leads_Operations Data Generation**

In [58]:
import pandas as pd
import numpy as np

np.random.seed(101)

lead_ids = df_dim_marketing_leads['LeadID'].tolist()
total_operations = len(lead_ids)

# 🆕 خريطة سريعة لوقت وصول كل Lead (من البلوك السابق)
lead_received_map = dict(zip(
    df_dim_marketing_leads['LeadID'],
    pd.to_datetime(df_dim_marketing_leads['ReceivedDateTime'])
))

# فترات عمل الموديريتور والسيلز أدمن
MODERATOR_WINDOWS = {
    'EMP_004': (pd.Timestamp('2023-02-01'), None),
    'EMP_021': (pd.Timestamp('2025-04-01'), pd.Timestamp('2025-10-31'))
}
SALES_ADMIN_WINDOWS = {
    'EMP_003': (pd.Timestamp('2023-01-15'), None),
    'EMP_020': (pd.Timestamp('2025-02-01'), None)
}

def pick_available_employee(windows, op_datetime):
    available = [eid for eid, (start, end) in windows.items() if start <= op_datetime and (end is None or op_datetime <= end)]
    if not available:
        available = [eid for eid, (start, end) in windows.items() if end is None]
    return np.random.choice(available)

# 🆕 دالة تزحزح أي وقت لأقرب وقت شغل فعلي (لو وقع خارج ساعات الشغل أو يوم إجازة)
def snap_to_next_working_slot(dt):
    current = dt
    for _ in range(30):
        if is_working_day(current.date()):
            if current.hour < WORK_START_HOUR:
                return current.replace(hour=WORK_START_HOUR, minute=random.randint(0, 59))
            elif current.hour >= WORK_END_HOUR:
                current = (current + pd.Timedelta(days=1)).replace(hour=WORK_START_HOUR, minute=random.randint(0, 59))
                continue
            else:
                return current
        else:
            current = (current + pd.Timedelta(days=1)).replace(hour=WORK_START_HOUR, minute=random.randint(0, 59))
    return current

statuses = ['Qualified / Assigned', 'Pending Unit Details', 'Unresponsive', 'Disqualified']
status_weights = [0.55, 0.15, 0.15, 0.15]
activities = ['First Call Screening', 'WhatsApp Unit Details Sent', 'Requirement Profiling', 'Re-engagement Attempt']
PROJECT_END_DATE = pd.Timestamp('2026-05-01')

operations_data = []

for i in range(total_operations):
    lead_id = lead_ids[i]
    lead_received = lead_received_map[lead_id]

    # مدة الاستجابة الأولية العشوائية (دقايق لمدة يومين تقريبًا)
    tentative_delay_min = np.random.randint(5, 2880)
    tentative_op_time = lead_received + pd.Timedelta(minutes=int(tentative_delay_min))

    # نزحزح الوقت لأقرب وقت شغل فعلي متاح
    op_datetime = snap_to_next_working_slot(tentative_op_time)
    op_datetime = min(op_datetime, PROJECT_END_DATE)

    # مدة الاستجابة الفعلية بعد التزحزح (بالدقايق، متسقة مع التوقيتين الحقيقيين)
    actual_response_minutes = max(int((op_datetime - lead_received).total_seconds() / 60), 1)

    mod_id = pick_available_employee(MODERATOR_WINDOWS, op_datetime)
    admin_id = pick_available_employee(SALES_ADMIN_WINDOWS, op_datetime)

    status = np.random.choice(statuses, p=status_weights)
    activity = np.random.choice(activities)

    operations_data.append({
        'OperationID': f"OP_{str(i+1).zfill(6)}",
        'LeadID': lead_id,
        'OperationDateTime': op_datetime.strftime('%Y-%m-%d %H:%M:%S'),
        'ModeratorID': mod_id,
        'SalesAdminID': admin_id,
        'LastActivity': activity,
        'OperationStatus': status,
        'ResponseTimeMinutes': actual_response_minutes,
        'UnitCatalogUpdated': True if status in ['Qualified / Assigned', 'Pending Unit Details'] else False
    })

df_fact_leads_operations = pd.DataFrame(operations_data)

print("=== Fact_Leads_Operations Sample Preview ===")
display(df_fact_leads_operations.head(10))
print(f"🎉 Generated {len(df_fact_leads_operations)} operation records!")

=== Fact_Leads_Operations Sample Preview ===


,OperationID,LeadID,OperationDateTime,ModeratorID,SalesAdminID,LastActivity,OperationStatus,ResponseTimeMinutes,UnitCatalogUpdated
0,OP_000001,LEAD_00001,2025-12-07 09:19:00,EMP_004,EMP_020,Re-engagement Attempt,Pending Unit Details,3206,True
1,OP_000002,LEAD_00002,2026-03-29 10:57:21,EMP_004,EMP_020,WhatsApp Unit Details Sent,Qualified / Assigned,604,True
2,OP_000003,LEAD_00003,2025-02-16 09:24:56,EMP_004,EMP_003,First Call Screening,Qualified / Assigned,2164,True
3,OP_000004,LEAD_00004,2025-05-11 09:37:43,EMP_004,EMP_003,First Call Screening,Disqualified,5975,False
4,OP_000005,LEAD_00005,2025-06-01 09:00:46,EMP_004,EMP_020,Requirement Profiling,Qualified / Assigned,3963,True
5,OP_000006,LEAD_00006,2024-08-04 09:54:26,EMP_004,EMP_003,Re-engagement Attempt,Qualified / Assigned,5003,True
6,OP_000007,LEAD_00007,2023-01-24 09:23:22,EMP_004,EMP_003,First Call Screening,Unresponsive,2930,False
7,OP_000008,LEAD_00008,2025-05-25 09:38:41,EMP_021,EMP_003,Re-engagement Attempt,Disqualified,2250,False
8,OP_000009,LEAD_00009,2025-06-23 09:25:58,EMP_021,EMP_003,WhatsApp Unit Details Sent,Qualified / Assigned,2184,True
9,OP_000010,LEAD_00010,2025-01-06 09:05:00,EMP_004,EMP_003,First Call Screening,Qualified / Assigned,2532,True


🎉 Generated 1200 operation records!


#### **2. Integrity Check:** LeadID, ModeratorID & SalesAdminID

In [59]:
# فحص: كل عملية لازم تكون بعد وقت وصول الـ Lead الخاص بها، وليس قبله
df_fact_leads_operations['OperationDateTime'] = pd.to_datetime(df_fact_leads_operations['OperationDateTime'])
invalid_timing = df_fact_leads_operations[
    df_fact_leads_operations.apply(lambda r: r['OperationDateTime'] < lead_received_map[r['LeadID']], axis=1)
]
print(f"🔍 Audit: عمليات معالجة قبل وصول الـ Lead = {len(invalid_timing)}")  # لازم يطلع صفر

🔍 Audit: عمليات معالجة قبل وصول الـ Lead = 0


#### **3. Live Google Sheets Export**

In [60]:
TAB_NAME_OPS = "Fact_Leads_Operations"

if folders:
    try:
        sh = gc.open(SPREADSHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(SPREADSHEET_NAME)
        drive_service.files().update(
            fileId=sh.id,
            addParents=folder_id,
            removeParents='root',
            fields='id, parents'
        ).execute()

    try:
        worksheet = sh.worksheet(TAB_NAME_OPS)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=TAB_NAME_OPS, rows="1500", cols="10")

    worksheet.clear()
    worksheet.update([df_fact_leads_operations.columns.values.tolist()] + df_fact_leads_operations.astype(str).values.tolist())

    print(f"🚀 Success! [{TAB_NAME_OPS}] exported successfully to [{SPREADSHEET_NAME}].")

🚀 Success! [Fact_Leads_Operations] exported successfully to [NC_RealEstate_DW_Data].


## **• Target Table:** Fact_Sales_Transactions •

#### **1. Fact_Sales_Transactions Data Generation**

In [61]:
import pandas as pd
import numpy as np

np.random.seed(2024)

# 1. تصفية الـ Leads المقبولة فقط من جدول العمليات الإدارية
qualified_leads = df_fact_leads_operations[
    df_fact_leads_operations['OperationStatus'] == 'Qualified / Assigned'
]['LeadID'].tolist()

# 2. فلترة العقارات: المعروضة للبيع (Sale Cash / Sale Installment) فقط وليس للإيجار
available_properties_df = df_dim_properties[
    df_dim_properties['SellingOption'].isin(['Sale Cash', 'Sale Installment'])
]
available_properties = available_properties_df['PropertyID'].tolist()

# عدد الصفقات = أقل قيمة بين (450، عدد الـ Leads المؤهلة، عدد العقارات المتاحة للبيع)
total_sales = min(450, len(qualified_leads), len(available_properties))

selected_leads = np.random.choice(qualified_leads, size=total_sales, replace=False)
selected_properties = np.random.choice(available_properties, size=total_sales, replace=False)

# 3. فترات عمل مندوبي المبيعات
SALES_REP_WINDOWS = {
    'EMP_002': (pd.Timestamp('2023-01-15'), None),
    'EMP_011': (pd.Timestamp('2024-02-01'), None),
    'EMP_012': (pd.Timestamp('2024-03-15'), pd.Timestamp('2025-02-28')),   # استقال
    'EMP_013': (pd.Timestamp('2024-05-01'), None),
    'EMP_014': (pd.Timestamp('2024-07-01'), pd.Timestamp('2024-12-31')),   # اتفصل
    'EMP_015': (pd.Timestamp('2024-09-01'), None),
    'EMP_017': (pd.Timestamp('2025-01-10'), None),
    'EMP_018': (pd.Timestamp('2025-03-01'), None),
    'EMP_019': (pd.Timestamp('2025-06-01'), None),
}

# خريطة سريعة لربط كل LeadID بالعميل الحقيقي بتاعه (من Dim_Marketing_Leads)
lead_to_client = dict(zip(df_dim_marketing_leads['LeadID'], df_dim_marketing_leads['ClientID']))

sales_data = []

for i in range(total_sales):
    lead_id = selected_leads[i]
    prop_id = selected_properties[i]

    prop_row = df_dim_properties[df_dim_properties['PropertyID'] == prop_id].iloc[0]
    op_row = df_fact_leads_operations[df_fact_leads_operations['LeadID'] == lead_id].iloc[0]
    op_date = pd.Timestamp(op_row['OperationDateTime'])

    days_to_close = np.random.randint(3, 46)
    sale_date = op_date + pd.Timedelta(days=days_to_close)

    if sale_date > pd.Timestamp('2026-05-01'):
        sale_date = pd.Timestamp('2026-04-28')

    rep_id = pick_available_employee(SALES_REP_WINDOWS, sale_date)

    # السعر الحقيقي للعقار = الدفعة المقدمة + المتبقي
    unit_price = prop_row['Amount_Downpayment'] + prop_row['RemainingAmount']

    company_commission_rate = np.random.choice([0.025, 0.030, 0.035, 0.040], p=[0.4, 0.3, 0.2, 0.1])
    gross_commission = unit_price * company_commission_rate

    rep_commission_rate = df_dim_employees.loc[df_dim_employees['EmployeeID'] == rep_id, 'CommissionRate'].values[0]
    agent_commission_egp = gross_commission * rep_commission_rate

    sales_data.append({
        'TransactionID': f"TXN_{str(i+1).zfill(6)}",
        'LeadID': lead_id,
        'ClientID': lead_to_client[lead_id],
        'PropertyID': prop_id,
        'SaleDate': sale_date.strftime('%Y-%m-%d'),
        'SalesRepID': rep_id,
        'TeamLeaderID': 'EMP_002',
        'UnitPrice_EGP': unit_price,
        'CompanyCommission_EGP': gross_commission,
        'AgentCommission_EGP': agent_commission_egp,
        'PaymentMethod': np.random.choice(['Installments', 'Cash / Lump Sum'], p=[0.85, 0.15])
    })

df_fact_sales_transactions = pd.DataFrame(sales_data)

# حساب IsFirstPurchase: أول صفقة بيع لكل عميل (بترتيب SaleDate) = 1، الباقي = 0
df_fact_sales_transactions['SaleDate'] = pd.to_datetime(df_fact_sales_transactions['SaleDate'])
first_purchase_dates = df_fact_sales_transactions.groupby('ClientID')['SaleDate'].transform('min')
df_fact_sales_transactions['IsFirstPurchase'] = (df_fact_sales_transactions['SaleDate'] == first_purchase_dates).astype(int)
df_fact_sales_transactions['SaleDate'] = df_fact_sales_transactions['SaleDate'].dt.strftime('%Y-%m-%d')

print("=== Fact_Sales_Transactions Sample Preview ===")
display(df_fact_sales_transactions.head(10))
print(f"🎉 Generated {len(df_fact_sales_transactions)} completed sales transactions (مع ClientID و IsFirstPurchase)!")

=== Fact_Sales_Transactions Sample Preview ===


,TransactionID,LeadID,ClientID,PropertyID,SaleDate,SalesRepID,TeamLeaderID,UnitPrice_EGP,CompanyCommission_EGP,AgentCommission_EGP,PaymentMethod,IsFirstPurchase
0,TXN_000001,LEAD_00944,CL_0000076,NC_1bA0000118,2025-09-10,EMP_018,EMP_002,16380000.0,409500.0,81900.0,Installments,0
1,TXN_000002,LEAD_00136,CL_0000116,NC_1dB0000173,2026-03-23,EMP_017,EMP_002,3320000.0,83000.0,16600.0,Installments,1
2,TXN_000003,LEAD_00003,CL_0000298,NC_1cB0000299,2025-03-22,EMP_018,EMP_002,10730000.0,321900.0,64380.0,Installments,1
3,TXN_000004,LEAD_00155,CL_0000001,NC_2bA0000455,2026-03-04,EMP_019,EMP_002,3680000.0,128800.0,25760.0,Installments,0
4,TXN_000005,LEAD_00585,CL_0000102,NC_1aB0000207,2025-11-26,EMP_015,EMP_002,22470000.0,674100.0,134820.0,Cash / Lump Sum,0
5,TXN_000006,LEAD_01076,CL_0000343,NC_2cA0000237,2025-12-08,EMP_002,EMP_002,23570000.0,589250.0,117850.0,Installments,1
6,TXN_000007,LEAD_00174,CL_0000174,NC_1bA0000190,2024-05-17,EMP_002,EMP_002,3270000.0,98100.0,19620.0,Cash / Lump Sum,1
7,TXN_000008,LEAD_00836,CL_0000263,NC_2aB0000264,2025-07-31,EMP_019,EMP_002,4340000.0,130200.0,26040.0,Installments,1
8,TXN_000009,LEAD_00352,CL_0000171,NC_2dB0000100,2026-01-14,EMP_013,EMP_002,14270000.0,356750.0,71350.0,Installments,1
9,TXN_000010,LEAD_00507,CL_0000281,NC_2dA0000013,2026-03-02,EMP_019,EMP_002,25410000.0,635250.0,127050.0,Installments,0


🎉 Generated 360 completed sales transactions (مع ClientID و IsFirstPurchase)!


#### **2. Integrity Check:** IsFirstPurchase = 1

In [62]:
first_purchase_check = df_fact_sales_transactions.groupby('ClientID')['IsFirstPurchase'].sum()
invalid_clients = first_purchase_check[first_purchase_check != 1]
print(f"🔍 Audit: عملاء بعدد IsFirstPurchase غلط (المفروض 1 بالظبط لكل عميل) = {len(invalid_clients)}")

🔍 Audit: عملاء بعدد IsFirstPurchase غلط (المفروض 1 بالظبط لكل عميل) = 0


#### **3. Live Google Sheets Export**

In [63]:
TAB_NAME_SALES = "Fact_Sales_Transactions"

if folders:
    try:
        sh = gc.open(SPREADSHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(SPREADSHEET_NAME)

    try:
        worksheet = sh.worksheet(TAB_NAME_SALES)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=TAB_NAME_SALES, rows="600", cols="12")

    worksheet.clear()
    worksheet.update([df_fact_sales_transactions.columns.values.tolist()] + df_fact_sales_transactions.astype(str).values.tolist())

    print(f"🚀 Success! [{TAB_NAME_SALES}] exported successfully to [{SPREADSHEET_NAME}].")

🚀 Success! [Fact_Sales_Transactions] exported successfully to [NC_RealEstate_DW_Data].


## **•Target Table:** Fact_RentalTransactions •

In [64]:
import pandas as pd
import numpy as np

np.random.seed(777)
PROJECT_END_DATE = pd.Timestamp('2026-05-01')

# 1. دالة توليد تاريخ إيجار موسمي (يرجّح 15 مايو - 15 سبتمبر)
def generate_seasonal_rental_date(entry_datetime):
    entry_datetime = pd.Timestamp(entry_datetime)
    for _ in range(50):
        candidate_year = random.choice([y for y in [2023, 2024, 2025, 2026] if y >= entry_datetime.year])

        # 75% احتمال يقع في موسم الذروة، 25% باقي السنة
        is_peak_season = random.random() < 0.75

        if is_peak_season:
            window_start = pd.Timestamp(f'{candidate_year}-05-15')
            window_end = pd.Timestamp(f'{candidate_year}-09-15')
        else:
            # نصف الاحتمال قبل الموسم (يناير-منتصف مايو)، ونصفه بعد الموسم (منتصف سبتمبر-ديسمبر)
            if random.random() < 0.5:
                window_start = pd.Timestamp(f'{candidate_year}-01-01')
                window_end = pd.Timestamp(f'{candidate_year}-05-14')
            else:
                window_start = pd.Timestamp(f'{candidate_year}-09-16')
                window_end = pd.Timestamp(f'{candidate_year}-12-31')

        if window_end < entry_datetime:
            continue
        actual_start = max(window_start, entry_datetime)
        if actual_start >= window_end:
            continue

        days_range = (window_end - actual_start).days
        candidate = actual_start + pd.Timedelta(days=random.randint(0, max(days_range, 0)))
        if candidate <= PROJECT_END_DATE:
            return candidate

    # احتياطي إذا لم توجد نافذة صالحة
    return min(entry_datetime + pd.Timedelta(days=30), PROJECT_END_DATE)

# 2. فلترة العقارات المؤجرة فعليًا فقط
rented_properties = df_dim_properties[
    (df_dim_properties['SellingOption'] == 'Rent') &
    (df_dim_properties['UnitStatus'] == 'Rented')
].copy()

# 3. استبعاد أي Lead استخدم بالفعل في صفقات البيع (Lead يتحول لنوع واحد فقط: بيع أو إيجار)
used_leads_in_sales = set(df_fact_sales_transactions['LeadID'])
qualified_leads_pool = df_fact_leads_operations[
    (df_fact_leads_operations['OperationStatus'] == 'Qualified / Assigned') &
    (~df_fact_leads_operations['LeadID'].isin(used_leads_in_sales))
]['LeadID'].tolist()

total_rentals = min(len(rented_properties), len(qualified_leads_pool))
selected_properties = rented_properties.sample(n=total_rentals, random_state=777).reset_index(drop=True)
selected_leads = np.random.choice(qualified_leads_pool, size=total_rentals, replace=False)

# 4. نفس فترات عمل مندوبي المبيعات (من بلوك Fact_Sales_Transactions، محفوظة في الجلسة)
rental_data = []
updated_rented_dates = {}

for i in range(total_rentals):
    prop_row = selected_properties.iloc[i]
    lead_id = selected_leads[i]
    entry_datetime = pd.Timestamp(prop_row['CreatedDateTime'])

    # تاريخ إيجار موسمي
    rental_date = generate_seasonal_rental_date(entry_datetime)
    updated_rented_dates[prop_row['PropertyID']] = rental_date.strftime('%Y-%m-%d %H:%M:%S')

    rep_id = pick_available_employee(SALES_REP_WINDOWS, rental_date)

    monthly_rent = prop_row['Amount_Downpayment']  # الإيجار الشهري المسجل وقت التوليد
    company_commission = monthly_rent * 2  # شهر من المالك + شهر من المستأجر
    rep_commission_rate = df_dim_employees.loc[df_dim_employees['EmployeeID'] == rep_id, 'CommissionRate'].values[0]
    agent_commission = company_commission * rep_commission_rate

    rental_data.append({
        'RentalTransactionID': f"RNT_{str(i+1).zfill(6)}",
        'LeadID': lead_id,
        'PropertyID': prop_row['PropertyID'],
        'RentalDate': rental_date.strftime('%Y-%m-%d'),
        'SalesRepID': rep_id,
        'TeamLeaderID': 'EMP_002',
        'MonthlyRent_EGP': monthly_rent,
        'CompanyCommission_EGP': company_commission,
        'AgentCommission_EGP': agent_commission
    })

df_fact_rental_transactions = pd.DataFrame(rental_data)

# 5. مزامنة RentedDateTime في Dim_Propertiesبالتواريخ الموسمية
df_dim_properties['RentedDateTime'] = df_dim_properties.apply(
    lambda r: updated_rented_dates.get(r['PropertyID'], r['RentedDateTime']), axis=1
)

print(f"🎉 Generated {len(df_fact_rental_transactions)} rental transactions (موسم مايو-سبتمبر مطبّق)!")
display(df_fact_rental_transactions.head(10))

🎉 Generated 37 rental transactions (موسم مايو-سبتمبر مطبّق)!


,RentalTransactionID,LeadID,PropertyID,RentalDate,SalesRepID,TeamLeaderID,MonthlyRent_EGP,CompanyCommission_EGP,AgentCommission_EGP
0,RNT_000001,LEAD_00807,NC_1dB0000193,2026-04-17,EMP_002,EMP_002,127000.0,254000.0,50800.0
1,RNT_000002,LEAD_00095,NC_2dA0000183,2025-07-25,EMP_017,EMP_002,135000.0,270000.0,54000.0
2,RNT_000003,LEAD_00225,NC_2aC0000261,2026-03-06,EMP_002,EMP_002,35000.0,70000.0,14000.0
3,RNT_000004,LEAD_00938,NC_1dA0000142,2026-05-01,EMP_018,EMP_002,41000.0,82000.0,16400.0
4,RNT_000005,LEAD_00630,NC_2cB0000500,2025-06-28,EMP_011,EMP_002,51000.0,102000.0,20400.0
5,RNT_000006,LEAD_01129,NC_1bA0000146,2026-04-23,EMP_013,EMP_002,48000.0,96000.0,19200.0
6,RNT_000007,LEAD_00415,NC_2bA0000016,2025-06-17,EMP_015,EMP_002,113000.0,226000.0,45200.0
7,RNT_000008,LEAD_00773,NC_1dB0000113,2024-06-12,EMP_013,EMP_002,57000.0,114000.0,22800.0
8,RNT_000009,LEAD_00980,NC_1cA0000335,2026-05-01,EMP_002,EMP_002,29000.0,58000.0,11600.0
9,RNT_000010,LEAD_00389,NC_2bA0000346,2025-08-17,EMP_015,EMP_002,144000.0,288000.0,57600.0


#### **2. Live Google Sheets Export**

In [65]:
TAB_NAME_RENT = "Fact_Rental_Transactions"

if folders:
    try:
        sh = gc.open(SPREADSHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(SPREADSHEET_NAME)

    try:
        worksheet = sh.worksheet(TAB_NAME_RENT)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=TAB_NAME_RENT, rows="600", cols="10")

    worksheet.clear()
    worksheet.update([df_fact_rental_transactions.columns.values.tolist()] + df_fact_rental_transactions.astype(str).values.tolist())
    print(f"🚀 Success! [{TAB_NAME_RENT}] exported successfully.")

🚀 Success! [Fact_Rental_Transactions] exported successfully.


## **• Target Table:** Fact_Monthly_Expenses •

#### **1. Fact_Monthly_Expenses Data Generation**

In [66]:
import calendar

# حساب عدد الموظفين العاملين فعليًا في أي شهر معين
emp_hire = pd.to_datetime(df_dim_employees['HireDate'])
emp_exit = df_dim_employees['ExitDate'].replace('', pd.NaT)
emp_exit = pd.to_datetime(emp_exit)

def get_active_headcount(period_end):
    active_mask = (emp_hire <= period_end) & (emp_exit.isna() | (emp_exit >= period_end))
    return active_mask.sum()

random.seed(555)
expense_records = []
expense_id_counter = 1

current_year, current_month = 2023, 1
end_year, end_month = 2026, 4

while (current_year, current_month) <= (end_year, end_month):
    last_day = calendar.monthrange(current_year, current_month)[1]
    period_end = pd.Timestamp(year=current_year, month=current_month, day=last_day)
    headcount = get_active_headcount(period_end)

    # 1. إيجار المكتب: شبه ثابت، يزيد تقريبًا 12% كل سنة (تضخم/نمو)
    base_rent = 15000 * (1.12 ** (current_year - 2023))
    rent_amount = round(base_rent, -2)

    # 2. الكهرباء: مرتبطة بعدد الموظفين الفعلي في الشهر
    electricity_amount = round(2000 + (headcount * random.randint(120, 180)), -1)

    # 3. مستلزمات مكتبية (استيكي نوتس، أقلام، مناديل، أكياس، مشروبات)
    supplies_amount = round(headcount * random.randint(60, 110), -1)

    for category, amount in [('Office Rent', rent_amount), ('Electricity', electricity_amount), ('Office Supplies', supplies_amount)]:
        expense_records.append({
            'ExpenseID': f"EXP_{expense_id_counter:05d}",
            'ExpenseYear': current_year,
            'ExpenseMonth': current_month,
            'ExpenseCategory': category,
            'Amount_EGP': amount
        })
        expense_id_counter += 1

    if current_month == 12:
        current_year += 1
        current_month = 1
    else:
        current_month += 1

df_fact_monthly_expenses = pd.DataFrame(expense_records)
print(f"🎉 Generated {len(df_fact_monthly_expenses)} monthly expense records across {df_fact_monthly_expenses['ExpenseCategory'].nunique()} categories!")
display(df_fact_monthly_expenses.head(12))

🎉 Generated 120 monthly expense records across 3 categories!


,ExpenseID,ExpenseYear,ExpenseMonth,ExpenseCategory,Amount_EGP
0,EXP_00001,2023,1,Office Rent,15000.0
1,EXP_00002,2023,1,Electricity,2400.0
2,EXP_00003,2023,1,Office Supplies,230.0
3,EXP_00004,2023,2,Office Rent,15000.0
4,EXP_00005,2023,2,Electricity,2520.0
5,EXP_00006,2023,2,Office Supplies,270.0
6,EXP_00007,2023,3,Office Rent,15000.0
7,EXP_00008,2023,3,Electricity,2620.0
8,EXP_00009,2023,3,Office Supplies,420.0
9,EXP_00010,2023,4,Office Rent,15000.0


#### **2. Live Google Sheets Export**

In [67]:
TAB_NAME_EXP = "Fact_Monthly_Expenses"

if folders:
    try:
        sh = gc.open(SPREADSHEET_NAME)
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(SPREADSHEET_NAME)

    try:
        worksheet = sh.worksheet(TAB_NAME_EXP)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=TAB_NAME_EXP, rows="150", cols="10")

    worksheet.clear()
    worksheet.update([df_fact_monthly_expenses.columns.values.tolist()] + df_fact_monthly_expenses.astype(str).values.tolist())

    print(f"🚀 Success! [{TAB_NAME_EXP}] exported successfully.")

🚀 Success! [Fact_Monthly_Expenses] exported successfully.


# **📄 Project Notebook: Synthetic Data Generation Engine**

### **1. Securely Checkingk:**
###### For ID Matches Between Tables.

In [68]:
# فحص مطابقة الـ IDs بين الجداول بصورة آمنة
try:
    invalid_properties = set(df_fact_sales_transactions['PropertyID']) - set(df_dim_properties['PropertyID'])
    invalid_leads = set(df_fact_sales_transactions['LeadID']) - set(df_dim_marketing_leads['LeadID'])
    invalid_reps = set(df_fact_sales_transactions['SalesRepID']) - set(df_dim_employees['EmployeeID'])

    print(f"🔍 Unmatched Properties: {len(invalid_properties)}")
    print(f"🔍 Unmatched Leads: {len(invalid_leads)}")
    print(f"🔍 Unmatched Sales Reps: {len(invalid_reps)}")

    if len(invalid_properties) == 0 and len(invalid_leads) == 0 and len(invalid_reps) == 0:
        print("✅ كل الـ IDs متطابقة 100% — البيانات جاهزة للرفع على SQL.")
    else:
        print("⚠️ فيه IDs غير متطابقة، راجعي خطوات التوليد قبل الرفع على SQL.")
except NameError as e:
    print(f"⚠️ متغير ناقص: {e} — تأكدي من تشغيل كل الخلايا السابقى بالترتيب.")

🔍 Unmatched Properties: 0
🔍 Unmatched Leads: 0
🔍 Unmatched Sales Reps: 0
✅ كل الـ IDs متطابقة 100% — البيانات جاهزة للرفع على SQL.


### **2. Create a Dedicated Export Folder CSV**

In [69]:
import os

output_folder = 'RealEstate_ETL_Files'
os.makedirs(output_folder, exist_ok=True)

files = {
    'Dim_Properties.csv': df_dim_properties,
    'Dim_Clients.csv': df_dim_clients,
    'Dim_Sellers.csv': df_dim_sellers,
    'Dim_Departments_Roles.csv': df_dim_departments_roles,
    'Dim_Employees.csv': df_dim_employees,
    'Dim_Employee_Career_History.csv': df_dim_employee_career_history,
    'Dim_Marketing_Campaigns.csv': df_dim_marketing_campaigns,   # 🆕
    'Dim_Marketing_Leads.csv': df_dim_marketing_leads,
    'Fact_Leads_Operations.csv': df_fact_leads_operations,
    'Fact_Sales_Transactions.csv': df_fact_sales_transactions,
    'Fact_Rental_Transactions.csv': df_fact_rental_transactions,
    'Fact_Monthly_Expenses.csv': df_fact_monthly_expenses          # 🆕
}

for filename, df in files.items():
    file_path = os.path.join(output_folder, filename)
    df.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f"✅ {filename} ({len(df)} صف) اتصدّرت بنجاح")

print(f"\n🎉 كل الجداول الـ 11 اتصدّرت في فولدر '{output_folder}'!")

✅ Dim_Properties.csv (500 صف) اتصدّرت بنجاح
✅ Dim_Clients.csv (500 صف) اتصدّرت بنجاح
✅ Dim_Sellers.csv (200 صف) اتصدّرت بنجاح
✅ Dim_Departments_Roles.csv (20 صف) اتصدّرت بنجاح
✅ Dim_Employees.csv (26 صف) اتصدّرت بنجاح
✅ Dim_Employee_Career_History.csv (38 صف) اتصدّرت بنجاح
✅ Dim_Marketing_Campaigns.csv (10 صف) اتصدّرت بنجاح
✅ Dim_Marketing_Leads.csv (1200 صف) اتصدّرت بنجاح
✅ Fact_Leads_Operations.csv (1200 صف) اتصدّرت بنجاح
✅ Fact_Sales_Transactions.csv (360 صف) اتصدّرت بنجاح
✅ Fact_Rental_Transactions.csv (37 صف) اتصدّرت بنجاح
✅ Fact_Monthly_Expenses.csv (120 صف) اتصدّرت بنجاح

🎉 كل الجداول الـ 11 اتصدّرت في فولدر 'RealEstate_ETL_Files'!
